# 1. Handling Libraries

In [3403]:
%pip install numpy
%pip install pandas
%pip install networkx
%pip install matplotlib
%pip install datashader
%pip install vl-convert-python
%pip install svgutils
%pip install lxml



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pi

In [3404]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from datashader.bundling import hammer_bundle
import svgutils.transform as st
from lxml import etree
import vl_convert as vlc
import xml.etree.ElementTree as ET
from collections import Counter
import json
import hashlib
from pathlib import Path
from IPython.display import display
import io
import re

pd.options.display.max_columns = None

# 2. Data Paths

In [3405]:
# nodes_url = "https://raw.githubusercontent.com/cns-iu/hra-ftu-vccf-supporting-information/main/code/data/asct-nodes.csv" 
# edges_url = "https://raw.githubusercontent.com/cns-iu/hra-ftu-vccf-supporting-information/main/code/data/asct-edges.csv"
# blood_edges_url = "https://raw.githubusercontent.com/cns-iu/hra-ftu-vccf-supporting-information/main/code/data/asct-blood-vasculature-edges.csv"

#use build_network.py to build data for latest release and save it in folder data
# data_version = 'v2.2'
# data_folder = f'All_Versions/data/data_{data_version}/'

# nodes_url = f"{data_folder}asct-nodes.csv" 
# edges_url = f"{data_folder}asct-edges.csv"
# blood_edges_url = f"{data_folder}asct-blood-vasculature-edges.csv"

# output_path = f'All_Versions/viz/viz_{data_version}/'

nodes_url = "data/v2.5/asct-nodes.csv" 
edges_url = "data/v2.5/asct-edges.csv"
# blood_edges_url = "data/asct-blood-vasculature-edges.csv"

# vessel_url = "data/Vessel.csv"
ftu_vessel_url = 'data/Table_S1.csv'

output_path = str(
    Path(
        "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator"
    )
)
print("output_path =", output_path)
## Make sure to save vega_config file in output_path/vega_config/

canvas_size = 1720

output_path = /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator


In [3406]:
vega_template = (
    Path(output_path)
    / "vega_config"
    / "vega_config_horizontal.json"
)

print("Template:", vega_template)
print("Exists:", vega_template.exists())

Template: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/vega_config/vega_config_horizontal.json
Exists: True


# 3. Algorithm and visualization logic

## 3.1 Process nodes and edges

In [3407]:
nodes = pd.read_csv(nodes_url)
edges = pd.read_csv(edges_url)

In [3408]:
omitted_organs = ['muscular_system', 'skeleton', 'lymph_vasculature', 'peripheral_nervous_system', 'blood_pelvis']

organ_order = ['trachea', 'main bronchus', 'respiratory system', 'mouth', 'heart', 'spinal cord', 'brain', 'eye', 'palatine tonsil', 
               'skin of body', 'thymus', 'lymph node', 'spleen',  'liver', 'Pancreas', 'small intestine',  'large intestine', 
               'kidney', 'urinary bladder', 'ureter', 'prostate gland',  'ovary', 'fallopian tube', 'uterus',  
               'placenta', 'knee', 'Bone marrow']

In [3409]:
nodes = nodes[~nodes['organ'].isin(omitted_organs)]
edges = edges[~edges['organ'].isin(omitted_organs)]

## 3.2 Edges refinement.

In [3410]:
## Detach respiratory system nodes from body and attach to lung.
respiratory_system_nodes = ['UBERON:0008886', 'UBERON:0004573', 'UBERON:0003920']

new_edges = edges[(edges['source']=='UBERON:0013702')&(edges['target'].isin(respiratory_system_nodes))].copy()
#connect the nodes to respiratory system instead of body
new_edges['source'].replace('UBERON:0013702', 'UBERON:0001004', inplace=True) 

#Update the edges
edges = edges[~((edges['source']=='UBERON:0013702')&(edges['target'].isin(respiratory_system_nodes)))]
edges = pd.concat([edges, new_edges])

/var/folders/3p/wpdcwlx57k93mldysxqtnprw0000gn/T/ipykernel_48726/771797898.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  new_edges['source'].replace('UBERON:0013702', 'UBERON:0001004', inplace=True)


In [3411]:
whole_graph = nx.from_pandas_edgelist(edges, source='source', target='target', edge_attr=True)
nx.is_tree(whole_graph)

True

In [3412]:
whole_graph

## 3.3 Nodes refinement

In [3413]:
def get_nodes(name=None, ontology_id=None):
    if name:
        return nodes[nodes['name'].str.lower()==name.lower()]
    if ontology_id:
        return nodes[nodes['ontology_id']==ontology_id]

In [3414]:
# organ_order_in_id = [get_nodes(name=organ)['id'].values[0] for organ in organ_order]

organ_order_in_id = []

for organ in organ_order:
    if len(get_nodes(name=organ)) > 0:
        organ_order_in_id += (list(get_nodes(name=organ)['id'].values))

In [3415]:
def get_organ_graph(organ_id):
    organ_graph = whole_graph.copy()
    organ_graph.remove_edge('UBERON:0013702', organ_id) #remove the edge between body and the organ

    connected_components = sorted(list(nx.connected_components(organ_graph)), key=len, reverse=False) #get the connected components

    return nx.subgraph(organ_graph, connected_components[0]) #return the smallest connected component

In [3416]:
#List of ontology ids of the organs that are branching out from body
organs_origin_body_id = list(nodes[nodes['ontology_id'].isin(list(nx.neighbors(whole_graph, 'UBERON:0013702')))]['ontology_id'])

#Get the updated list of organ ids in order containing only ids that are connected to the body node.
new_organ_order_in_id = []
for organ_id in organ_order_in_id:
    if organ_id in organs_origin_body_id:
        new_organ_order_in_id.append(organ_id)

organ_order_in_id = new_organ_order_in_id

In [3417]:
## Generate a new integer-based ID, that takes into account the organ order
#(This is necesary for the layout, later we can order the nodes and edges based on this integer ID, and then vega will visualize the organs in the desired order)

id_to_graph_int_id = {'UBERON:0013702': 0}

next_id = 1

for i, organ_id in enumerate(organ_order_in_id):
    branch_graph = get_organ_graph(organ_id)

    branch_graph_new_ids = dict(zip(nx.dfs_preorder_nodes(branch_graph, source=organ_id), range(next_id, next_id+branch_graph.number_of_nodes())))

    id_to_graph_int_id.update(branch_graph_new_ids)

    next_id += branch_graph.number_of_nodes()


In [3418]:
nodes['graph_int_id'] = nodes['id'].map(id_to_graph_int_id)

In [3419]:
edges['source_int'] = edges['source'].map(id_to_graph_int_id)
edges['target_int'] = edges['target'].map(id_to_graph_int_id)

In [3420]:
whole_graph_int = nx.from_pandas_edgelist(edges, source='source_int', target='target_int', edge_attr=True)

In [3421]:
graph_int_id_to_id = {v: k for k, v in id_to_graph_int_id.items()}

In [3422]:
# id_to_graph_int_id

## 3.4 Testing out nodes and edges (Skippable)

In [3423]:
edges['organ'].value_counts()

organ
blood_vasculature    4791
ovary                2070
large_intestine      1330
allen_brain           964
uterus                696
heart                 298
lung                  211
spleen                199
small_intestine       188
fallopian_tube        175
pancreas              175
placenta              166
kidney                157
lymph_node            128
spinal_cord           127
thymus                106
mouth                 103
skin                   94
eye                    85
palatine_tonsil        70
bone_marrow            62
liver                  52
trachea                42
prostate               34
urinary_bladder        32
main_bronchus          31
knee                   29
ureter                 21
Name: count, dtype: int64

In [3424]:
edges

,organ,source,target,source_type,target_type,source_int,target_int
0,blood_vasculature,UBERON:0013702,UBERON:0000948,AS,AS,0.0,380.0
1,allen_brain,UBERON:0013702,UBERON:0000955,AS,AS,0.0,5605.0
2,allen_brain,UBERON:0013702,CL:0000115,AS,CT,0.0,NaN
3,allen_brain,UBERON:0013702,CL:0000669,AS,CT,0.0,NaN
4,allen_brain,UBERON:0013702,CL:0000542,AS,CT,0.0,NaN
...,...,...,...,...,...,...,...
12464,uterus,UBERON:0012276$$9067,ASCTB-TEMP:glandular-epithelial-cell$$9068,AS,CT,12166.0,12167.0
12465,uterus,UBERON:0011949$$9069,ASCTB-TEMP:luminal-epithelial-cell$$9070,AS,CT,12168.0,12169.0
14,lung,UBERON:0001004,UBERON:0008886,AS,AS,66.0,253.0
15,lung,UBERON:0001004,UBERON:0004573,AS,AS,66.0,265.0


In [3425]:
nodes.duplicated(subset=['id', 'type', 'organ', 'ontology_id']).sum()

np.int64(0)

In [3426]:
nodes.duplicated(subset=['id'], keep=False).sum()

np.int64(0)

In [3427]:
nodes

,id,name,type,organ,ontology_id,graph_int_id
0,UBERON:0013702,body,AS,body,UBERON:0013702,0.0
1,UBERON:0000948,heart,AS,blood_vasculature,UBERON:0000948,380.0
2,UBERON:0002079,left cardiac atrium,AS,blood_vasculature,UBERON:0002079,381.0
3,CL:0010008,cardiac endothelial cell,CT,blood_vasculature,CL:0010008,382.0
4,FMA:49913,Left inferior pulmonary vein,AS,blood_vasculature,FMA:49913,383.0
...,...,...,...,...,...,...
12462,UBERON:0002451$$9066,endometrial gland,AS,uterus,UBERON:0002451,12165.0
12463,UBERON:0012276$$9067,endometrium glandular epithelium,AS,uterus,UBERON:0012276,12166.0
12464,ASCTB-TEMP:glandular-epithelial-cell$$9068,glandular epithelial cell,CT,uterus,ASCTB-TEMP:glandular-epithelial-cell,12167.0
12465,UBERON:0011949$$9069,endometrium luminal epithelium,AS,uterus,UBERON:0011949,12168.0


In [3428]:
nodes.isna().sum()

id              0
name            1
type            0
organ           0
ontology_id     0
graph_int_id    4
dtype: int64

In [3429]:
nodes.loc[nodes['name'].isna()]

,id,name,type,organ,ontology_id,graph_int_id
7931,ALBUMIN,NaN,AS,liver,ALBUMIN,7255.0


In [3430]:
# Nodes that are branching from the 'body' node

nodes[nodes['ontology_id'].isin(list(nx.neighbors(whole_graph, 'UBERON:0013702')))]

,id,name,type,organ,ontology_id,graph_int_id
1,UBERON:0000948,heart,AS,blood_vasculature,UBERON:0000948,380.0
4792,UBERON:0000955,brain,AS,allen_brain,UBERON:0000955,5605.0
5749,CL:0000115,Endothelial cell,CT,allen_brain,CL:0000115,NaN
5753,CL:0000669,pericyte,CT,allen_brain,CL:0000669,NaN
5755,CL:0000542,lymphocyte,CT,allen_brain,CL:0000542,NaN
...,...,...,...,...,...,...
12286,CL:0000115$$8892,endothelial cell,CT,uterus,CL:0000115,11729.0
12305,CL:0000542$$8911,lymphocyte,CT,uterus,CL:0000542,11937.0
12313,CL:0000115$$8919,endothelial cell,CT,uterus,CL:0000115,11952.0
12332,CL:0000542$$8938,lymphocyte,CT,uterus,CL:0000542,12160.0


In [3431]:
id_to_graph_int_id['UBERON:0013702']

0

In [3432]:
graph_int_id_to_id[0]

'UBERON:0013702'

In [3433]:
nodes[nodes['organ'] == 'blood_vasculature']

,id,name,type,organ,ontology_id,graph_int_id
1,UBERON:0000948,heart,AS,blood_vasculature,UBERON:0000948,380.0
2,UBERON:0002079,left cardiac atrium,AS,blood_vasculature,UBERON:0002079,381.0
3,CL:0010008,cardiac endothelial cell,CT,blood_vasculature,CL:0010008,382.0
4,FMA:49913,Left inferior pulmonary vein,AS,blood_vasculature,FMA:49913,383.0
5,CL:0000071,blood vessel endothelial cell,CT,blood_vasculature,CL:0000071,384.0
...,...,...,...,...,...,...
4787,FMA:8619,Right posterior segmental artery,AS,blood_vasculature,FMA:8619,5333.0
4788,CL:0000071$$3651,blood vessel endothelial cell,CT,blood_vasculature,CL:0000071,5334.0
4789,CL:1000413$$3652,endothelial cell of artery,CT,blood_vasculature,CL:1000413,5335.0
4790,CL:0000057$$3653,fibroblast,CT,blood_vasculature,CL:0000057,5336.0


In [3434]:
edges[edges['organ'] == 'blood_vasculature']

,organ,source,target,source_type,target_type,source_int,target_int
0,blood_vasculature,UBERON:0013702,UBERON:0000948,AS,AS,0.0,380.0
34,blood_vasculature,UBERON:0000948,UBERON:0002079,AS,AS,380.0,381.0
35,blood_vasculature,UBERON:0000948,UBERON:0002084,AS,AS,380.0,464.0
36,blood_vasculature,UBERON:0000948,UBERON:0002078,AS,AS,380.0,3907.0
37,blood_vasculature,UBERON:0000948,UBERON:0002080,AS,AS,380.0,5157.0
...,...,...,...,...,...,...,...
4844,blood_vasculature,FMA:8617,CL:0000359$$3650,AS,CT,5328.0,5332.0
4845,blood_vasculature,FMA:8619,CL:0000071$$3651,AS,CT,5333.0,5334.0
4846,blood_vasculature,FMA:8619,CL:1000413$$3652,AS,CT,5333.0,5335.0
4847,blood_vasculature,FMA:8619,CL:0000057$$3653,AS,CT,5333.0,5336.0


## 3.5 Removal of the blood vasculature nodes from the whole (partonomy) graph
We only keep those that are necessary to keep the network connected

In [3435]:
# remove those blood nodes from the graph, which if we remove, the graph will be still connected, repeat until no such node is found.
blood_nodes_candidates = set(nodes[nodes['organ']=='blood_vasculature'].graph_int_id.tolist())

distance_from_body = nx.single_source_shortest_path_length(whole_graph_int, source=0)

distance_from_body_blood_nodes = {k: v for k, v in distance_from_body.items() if k in blood_nodes_candidates}

node_visiting_order = sorted(distance_from_body_blood_nodes.items(), key=lambda x: x[1], reverse=True)

new_truncated_graph = whole_graph_int.copy()

for node in node_visiting_order:
    new_truncated_graph.remove_node(node[0])
    if nx.is_connected(new_truncated_graph):
        blood_nodes_candidates.remove(node[0])
    else:
        break

nodes['keep'] = nodes.apply(lambda row: True if (row['organ']!='blood_vasculature' or row['graph_int_id'] in blood_nodes_candidates) else False, axis=1)

In [3436]:
truncated_nodes = nodes[nodes['keep']]
truncated_edges = edges[edges['source_int'].isin(truncated_nodes['graph_int_id']) | edges['target_int'].isin(truncated_nodes['graph_int_id'])]
truncated_graph = nx.from_pandas_edgelist(truncated_edges, source='source_int', target='target_int', edge_attr=True)

In [3437]:
missing_organ_names = [
    "liver",
    "large intestine",
    "small intestine",
    "lung",
]

for organ_name in missing_organ_names:
    print("\n" + "=" * 70)
    print(organ_name.upper())
    print("=" * 70)

    matches = truncated_nodes[
        truncated_nodes["name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(organ_name.lower())
    ][
        [
            "graph_int_id",
            "id",
            "name",
            "organ",
            "type",
            "ontology_id",
        ]
    ]

    display(matches)


LIVER


,graph_int_id,id,name,organ,type,ontology_id
7922,7246.0,UBERON:0002107,liver,liver,AS,UBERON:0002107



LARGE INTESTINE


,graph_int_id,id,name,organ,type,ontology_id
6592,7661.0,UBERON:0000059,large intestine,large_intestine,AS,UBERON:0000059



SMALL INTESTINE


,graph_int_id,id,name,organ,type,ontology_id
11064,7473.0,UBERON:0002108,small intestine,small_intestine,AS,UBERON:0002108



LUNG


,graph_int_id,id,name,organ,type,ontology_id
8072,164.0,UBERON:0002048,lung,lung,AS,UBERON:0002048


In [3438]:
# ============================================================
# AUTHORITATIVE TABLE_S1 FTU INVENTORY
# ============================================================

table_s1_ftu = pd.read_csv(ftu_vessel_url)

TABLE_S1_FTU_DEFINITIONS = (
    table_s1_ftu[["FTU", "FTUID"]]
    .dropna(subset=["FTU", "FTUID"])
    .assign(
        FTU=lambda frame: (
            frame["FTU"].astype(str).str.strip()
        ),
        FTUID=lambda frame: (
            frame["FTUID"].astype(str).str.strip()
        ),
    )
    .drop_duplicates(subset=["FTU", "FTUID"])
    .sort_values(["FTU", "FTUID"])
    .reset_index(drop=True)
)

if TABLE_S1_FTU_DEFINITIONS["FTU"].nunique() != 28:
    raise ValueError("Expected 28 unique Table_S1 FTU names")

if TABLE_S1_FTU_DEFINITIONS["FTUID"].nunique() != 28:
    raise ValueError("Expected 28 unique Table_S1 FTU ontology IDs")

TABLE_S1_FTU_ID_BY_NAME = {
    str(row.FTU).strip().lower(): str(row.FTUID).strip()
    for row in TABLE_S1_FTU_DEFINITIONS.itertuples(index=False)
}

# Table_S1 uses the generic crypt ID UBERON:0001983, while the
# source graph represents the requested large-intestine subtype as
# UBERON:0001984. This is a source-established graph alias, not an
# invented FTU relationship.
TABLE_S1_GRAPH_FTU_ID_OVERRIDES = {
    "crypt of lieberkuhn": "UBERON:0001984",
}

FTUs = sorted(
    TABLE_S1_FTU_DEFINITIONS["FTUID"].unique().tolist()
)

CONFIRMED_GRAPH_FTU_IDS = (
    set(FTUs)
    | set(TABLE_S1_GRAPH_FTU_ID_OVERRIDES.values())
)

GRAPH_FTU_ID_TO_TABLE_S1_ID = {
    ftu_id: ftu_id
    for ftu_id in FTUs
}

for ftu_name, graph_ftu_id in (
    TABLE_S1_GRAPH_FTU_ID_OVERRIDES.items()
):
    GRAPH_FTU_ID_TO_TABLE_S1_ID[graph_ftu_id] = (
        TABLE_S1_FTU_ID_BY_NAME[ftu_name]
    )

print(
    "Table_S1 FTU inventory:",
    TABLE_S1_FTU_DEFINITIONS["FTU"].nunique(),
    "unique names /",
    TABLE_S1_FTU_DEFINITIONS["FTUID"].nunique(),
    "unique ontology IDs"
)


Table_S1 FTU inventory: 28 unique names / 28 unique ontology IDs


In [3439]:
Main_Organs = list(nodes[nodes['ontology_id'].isin(list(nx.neighbors(whole_graph, 'UBERON:0013702')))]['id'])

## 3.6 Statistics for Male and Female Graphs (Skippable)

In [3440]:
def process_nodes(nodes, truncated_nodes, truncated_edges, FTUs, gender='female'):
    """
    Process nodes based on gender and FTU inclusion.

    Parameters:
        nodes (DataFrame): The input nodes DataFrame.
        truncated_nodes (DataFrame): Nodes DataFrame after truncation.
        truncated_edges (DataFrame): Edges DataFrame after truncation.
        FTUs (set): Set of FTU ontology IDs.
        gender (str): Gender type ('female', 'male', 'full').

    Returns:
        dict: A dictionary with FTU counters and filtered DataFrame.
    """

    # Gender-specific filtering
    if gender == 'female':
        nodes_filtered = nodes[nodes['organ'] != 'prostate']
    elif gender == 'male':
        exclude_organs = ['fallopian_tube', 'ovary', 'uterus', 'placenta']
        nodes_filtered = nodes[~nodes['organ'].isin(exclude_organs)]
    else:
        nodes_filtered = nodes

    # FTU lists
    ftu_list = [oid for oid in nodes_filtered['ontology_id'] if oid in FTUs]
    ftu_name_list = [row['name'] for _, row in nodes_filtered.iterrows() if row['ontology_id'] in FTUs]

    # Counters for ontology IDs and names
    ftu_dict = Counter(ftu_list)
    ftu_dict_name = Counter(ftu_name_list)

    # Print FTU names counter
    print(f"FTU names and count ({gender}):\n", ftu_dict_name)

    # Conditions
    condition1 = nodes_filtered['keep'] == True
    condition2 = nodes_filtered['type'] == 'AS'
    condition3 = nodes_filtered['type'] == 'CT'
    condition4 = nodes_filtered['ontology_id'].isin(FTUs)

    # Printing counts based on conditions
    print(f"\nCount of nodes (AS) ({gender}):", len(nodes_filtered[condition1 & condition2]), len(nodes_filtered[condition1 & condition2]['ontology_id'].unique()))
    print(f"\nCount of nodes (CT) ({gender}):", len(nodes_filtered[condition1 & condition3]), len(nodes_filtered[condition1 & condition3]['ontology_id'].unique()))
    print(f"\nCount of nodes (FTUs) ({gender}):", len(nodes_filtered[condition1 & condition4]), len(nodes_filtered[condition1 & condition4]['ontology_id'].unique()))

    # Sorting nodes and edges
    nodes_df = truncated_nodes.sort_values('graph_int_id').copy()
    edges_df = truncated_edges.sort_values('target_int').copy()

    # Create graph and calculate parents
    graph = nx.from_pandas_edgelist(edges_df, source='source_int', target='target_int', edge_attr=True)
    parents_dict = dict(nx.bfs_predecessors(graph, source=380))
    
    # Add parent to nodes DataFrame
    #nodes_df['parent'] = nodes_df['graph_int_id'].map(parents_dict).fillna(0).astype(int)
    nodes_df['parent'] = nodes_df['graph_int_id'].apply(lambda x: parents_dict.get(x, 0)).astype(int)

    # Additional filtering for gender-specific nodes
    if gender == 'female':
        nodes_df = nodes_df[nodes_df['organ'] != 'prostate']
    elif gender == 'male':
        nodes_df = nodes_df[~nodes_df['organ'].isin(exclude_organs)]

    # Drop duplicates and reset index
    newdf = nodes_df.drop_duplicates(subset=['parent', 'graph_int_id'], keep='first').reset_index(drop=True)
    print(f"\nNumber of structural edges ({gender}):", len(newdf) - 1)

    return {
        'ftu_dict': ftu_dict,
        'ftu_dict_name': ftu_dict_name,
        'filtered_df': newdf
    }

In [3441]:
'''li_nodes = organ_nodes[
    organ_nodes["organ"]
    .astype(str)
    .str.lower()
    .eq("large_intestine")
].copy()

li_ftus = li_nodes[
    li_nodes["ontology_id"]
    .astype(str)
    .str.strip()
    .isin(ftu_set)
].copy()

print("Large intestine FTU rows:", len(li_ftus))

display(
    li_ftus[
        [
            "graph_int_id",
            "name",
            "ontology_id",
            "parent"
        ]
    ].sort_values(
        ["ontology_id", "graph_int_id"]
    )
)
'''

'li_nodes = organ_nodes[\n    organ_nodes["organ"]\n    .astype(str)\n    .str.lower()\n    .eq("large_intestine")\n].copy()\n\nli_ftus = li_nodes[\n    li_nodes["ontology_id"]\n    .astype(str)\n    .str.strip()\n    .isin(ftu_set)\n].copy()\n\nprint("Large intestine FTU rows:", len(li_ftus))\n\ndisplay(\n    li_ftus[\n        [\n            "graph_int_id",\n            "name",\n            "ontology_id",\n            "parent"\n        ]\n    ].sort_values(\n        ["ontology_id", "graph_int_id"]\n    )\n)\n'

In [3442]:
result_female = process_nodes(nodes, truncated_nodes, truncated_edges, FTUs, 'female')


FTU names and count (female):
 Counter({'Pancreatic acinus': 6, 'islet of Langerhans': 6, 'bronchus submucosal gland': 4, 'intestinal villus': 4, 'cortical collecting duct': 2, 'thick ascending limb of loop of Henle': 2, 'descending limb of loop of Henle': 2, 'hair follicle': 2, 'alveolus of lung': 2, 'dermal papilla': 2, 'nephron': 1, 'Renal Corpuscle': 1, 'loop of Henle ascending limb thin segment': 1, 'outer medullary collecting duct': 1, 'inner medullary collecting duct': 1, 'liver lobule': 1, 'dentogingival junction': 1, 'parotid gland ducto-acinar unit': 1, 'sublingual gland ducto-acinar unit': 1, 'submandibular gland ducto-acinar unit': 1, 'chorionic terminal villus': 1, 'epidermal ridge of digit': 1, 'red pulp of spleen': 1, 'white pulp of spleen': 1, 'thymus lobule': 1})

Count of nodes (AS) (female): 4208 2162

Count of nodes (CT) (female): 8195 1211

Count of nodes (FTUs) (female): 47 25

Number of structural edges (female): 12399


In [3443]:
result_male = process_nodes(nodes, truncated_nodes, truncated_edges, FTUs, 'male')

FTU names and count (male):
 Counter({'Pancreatic acinus': 6, 'islet of Langerhans': 6, 'bronchus submucosal gland': 4, 'intestinal villus': 4, 'cortical collecting duct': 2, 'thick ascending limb of loop of Henle': 2, 'descending limb of loop of Henle': 2, 'hair follicle': 2, 'alveolus of lung': 2, 'dermal papilla': 2, 'nephron': 1, 'Renal Corpuscle': 1, 'loop of Henle ascending limb thin segment': 1, 'outer medullary collecting duct': 1, 'inner medullary collecting duct': 1, 'liver lobule': 1, 'dentogingival junction': 1, 'parotid gland ducto-acinar unit': 1, 'sublingual gland ducto-acinar unit': 1, 'submandibular gland ducto-acinar unit': 1, 'Prostate Glandular Acinus': 1, 'epidermal ridge of digit': 1, 'red pulp of spleen': 1, 'white pulp of spleen': 1, 'thymus lobule': 1})

Count of nodes (AS) (male): 2523 1943

Count of nodes (CT) (male): 6807 1177

Count of nodes (FTUs) (male): 47 25

Number of structural edges (male): 9326


In [3444]:
result_full = process_nodes(nodes, truncated_nodes, truncated_edges, FTUs, 'full')

FTU names and count (full):
 Counter({'Pancreatic acinus': 6, 'islet of Langerhans': 6, 'bronchus submucosal gland': 4, 'intestinal villus': 4, 'cortical collecting duct': 2, 'thick ascending limb of loop of Henle': 2, 'descending limb of loop of Henle': 2, 'hair follicle': 2, 'alveolus of lung': 2, 'dermal papilla': 2, 'nephron': 1, 'Renal Corpuscle': 1, 'loop of Henle ascending limb thin segment': 1, 'outer medullary collecting duct': 1, 'inner medullary collecting duct': 1, 'liver lobule': 1, 'dentogingival junction': 1, 'parotid gland ducto-acinar unit': 1, 'sublingual gland ducto-acinar unit': 1, 'submandibular gland ducto-acinar unit': 1, 'chorionic terminal villus': 1, 'Prostate Glandular Acinus': 1, 'epidermal ridge of digit': 1, 'red pulp of spleen': 1, 'white pulp of spleen': 1, 'thymus lobule': 1})

Count of nodes (AS) (full): 4222 2173

Count of nodes (CT) (full): 8215 1224

Count of nodes (FTUs) (full): 48 26

Number of structural edges (full): 12433


## 3.7 Construct the Vega Viz files

In [3445]:
print(
    truncated_nodes[
        truncated_nodes["graph_int_id"].isin(node_ids)
    ]["organ"]
    .value_counts()
)

organ
skin    94
body     1
Name: count, dtype: int64


In [3446]:
def update_vega_config(vega_config_names, filename):
    for item in vega_config_names['data'][0]['values']:
        if item['ontology_id'] not in FTUs and item['ontology_id'] not in Main_Organs:
            item['name'] = ''
        
    with open(f"{output_path}/butterfly_names_{filename}_vega_viz_updated.json", "w") as outfile:
        outfile.write(json.dumps(vega_config_names, indent=4))
        print(f'File saved as "{output_path}/butterfly_names_{filename}_vega_viz_updated.json"')

    return vega_config_names

In [3447]:
def construct_network_create_vega_viz(
    nodes_dataframe,
    edges_dataframe,
    filename="butterfly",
    root_id=0,
    only_female=False,
    only_male=False,
    scenegraph=False,
    show_labels=False
):
    """
    Construct an organ/tree Vega configuration.

    Colors:
      Red    = FTU
      Purple = Anatomical Structure (AS)
      Orange = Cell Type / other non-AS nodes

    Labels are controlled by the `show_label` column.
    """

    # -------------------------------------------------
    # Copy and sort input data
    # -------------------------------------------------
    nodes_df = (
        nodes_dataframe
        .sort_values("graph_int_id")
        .copy()
    )

    edges_df = (
        edges_dataframe
        .sort_values("target_int")
        .copy()
    )

    # -------------------------------------------------
    # Build graph
    # -------------------------------------------------
    graph = nx.from_pandas_edgelist(
        edges_df,
        source="source_int",
        target="target_int",
        edge_attr=True,
        create_using=nx.DiGraph()
    )

    # -------------------------------------------------
    # Determine parent of each node
    # -------------------------------------------------
    parents_dict = dict(
        nx.bfs_predecessors(
            graph,
            source=root_id
        )
    )

    nodes_df["parent"] = (
        nodes_df["graph_int_id"]
        .apply(
            lambda x: parents_dict.get(x, 0)
        )
        .astype(int)
    )

    # -------------------------------------------------
    # Normalize FTU IDs
    # -------------------------------------------------
    ftu_set = {
        str(x).strip()
        for x in CONFIRMED_GRAPH_FTU_IDS
    }

    ontology_ids = (
        nodes_df["ontology_id"]
        .astype(str)
        .str.strip()
    )

    # -------------------------------------------------
    # Node colors
    # -------------------------------------------------
    nodes_df["color"] = np.where(
        ontology_ids.isin(ftu_set),
        "#C62828",       # FTU = red
        np.where(
            nodes_df["type"].eq("AS"),
            "#984ea0",   # AS = purple
            "#ff7f00"    # CT / other = orange
        )
    )

    # -------------------------------------------------
    # Preserve explicit caller-provided label choices.
    # If no label column was supplied, show FTUs only
    # when show_labels=True.
    # -------------------------------------------------
    if "show_label" in nodes_df.columns:
        nodes_df["show_label"] = (
            pd.to_numeric(
                nodes_df["show_label"],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
        )
    elif show_labels:
        nodes_df["show_label"] = (
            ontology_ids.isin(ftu_set)
        ).astype(int)
    else:
        nodes_df["show_label"] = 0

    # -------------------------------------------------
    # Clean names
    # -------------------------------------------------
    nodes_df["name"] = (
        nodes_df["name"]
        .fillna("")
        .astype(str)
        .str.capitalize()
    )

    # -------------------------------------------------
    # Vega uses graph_int_id as node id
    # -------------------------------------------------
    nodes_df.rename(
        columns={
            "id": "id_from_ontology_id",
            "graph_int_id": "id"
        },
        inplace=True
    )

    # -------------------------------------------------
    # Scenegraph mode
    # -------------------------------------------------
    if scenegraph:

        nodes_df.rename(
            columns={"name": "name_label"},
            inplace=True
        )

        nodes_df["name"] = (
            nodes_df["id"].copy()
        )

    # -------------------------------------------------
    # Female / male filtering
    # -------------------------------------------------
    if only_female and not only_male:

        nodes_df = nodes_df[
            nodes_df["organ"] != "prostate"
        ]

        fn = filename + "_female"

    elif only_male and not only_female:

        nodes_df = nodes_df[
            ~nodes_df["organ"].isin(
                [
                    "fallopian_tube",
                    "ovary",
                    "uterus",
                    "placenta"
                ]
            )
        ]

        fn = filename + "_male"

    elif not only_female and not only_male:

        fn = filename + "_full"

    else:

        raise ValueError(
            "The parameters `only_female` and "
            "`only_male` cannot both be True."
        )

    # -------------------------------------------------
    # Convert nodes to Vega data
    # -------------------------------------------------
    nodes_json = nodes_df[
        [
            "id",
            "name",
            "parent",
            "type",
            "ontology_id",
            "id_from_ontology_id",
            "color",
            "organ",
            "show_label"
        ]
    ].to_dict(
        orient="index"
    )

    # -------------------------------------------------
    # Root must not have a parent
    # -------------------------------------------------
    for _, node in nodes_json.items():

        if int(node["id"]) == int(root_id):

            node.pop(
                "parent",
                None
            )

            break

    nodes_json = list(
        nodes_json.values()
    )

    # -------------------------------------------------
    # Load Vega template
    # -------------------------------------------------
    vega_config_path = (
        Path(output_path)
        / "vega_config"
        / "vega_config_horizontal.json"
    )

    if not vega_config_path.exists():

        raise FileNotFoundError(
            f"Vega template not found: "
            f"{vega_config_path}"
        )

    with open(
        vega_config_path,
        encoding="utf8"
    ) as json_file:

        config = json.load(
            json_file
        )

    # -------------------------------------------------
    # Use a compact canvas sized to the selected graph.
    # The tree stays near the upper-left with a small
    # margin; width follows graph depth and height follows
    # the number of terminal branches.
    # -------------------------------------------------
    depth_by_node = nx.single_source_shortest_path_length(
        graph,
        root_id
    )

    max_depth = max(
        (
            depth_by_node.get(int(node["id"]), 0)
            for node in nodes_json
        ),
        default=1
    )

    leaf_count = max(
        1,
        sum(
            graph.out_degree(int(node["id"])) == 0
            for node in nodes_json
        )
    )

    # Use a true compact hierarchy layout: every successive depth is
    # 100 px apart. Angled labels keep the tighter levels readable.
    # The right margin is reserved for readable labels; fonts are unchanged.
    level_spacing = 100
    tree_left_margin = 50
    tree_right_label_margin = 260
    compact_tree_width = level_spacing * max_depth

    config["width"] = int(
        max(
            800,
            min(2000, tree_left_margin + compact_tree_width + tree_right_label_margin)
        )
    )

    # Keep every organ within the landscape-page height used
    # successfully by Kidney. Dense unlabeled nodes may overlap;
    # the important Body/organ/FTU labels retain their spacing.
    base_tree_height = int(
        max(
            320,
            min(650, 100 + 22 * leaf_count)
        )
    )
    rotated_label_clearance = 180
    config["height"] = base_tree_height + rotated_label_clearance

    config["padding"] = 0

    tree_data = config["data"][0]

    for transform in tree_data.get("transform", []):

        if transform.get("type") == "tree":
            transform["size"] = [
                max(150, base_tree_height - 80),
                compact_tree_width
            ]

        elif transform.get("as") == "x_draw":
            transform["expr"] = f"datum.y + {tree_left_margin}"

        elif transform.get("as") == "y_draw":
            transform["expr"] = "datum.x + 220"

    for mark in config.get("marks", []):

        if mark.get("type") == "path":
            mark["encode"]["update"]["strokeWidth"] = {
                "value": 1.1
            }

        elif mark.get("type") == "symbol":
            mark["encode"]["enter"]["size"] = {
                "value": 45
            }
            mark["encode"]["enter"]["strokeWidth"] = {
                "value": 0.5
            }

        elif mark.get("type") == "text":
            mark["encode"]["enter"]["fontSize"] = {
                "value": 12
            }

    # -------------------------------------------------
    # Insert data
    # -------------------------------------------------
    config["data"][0]["values"] = nodes_json

    # -------------------------------------------------
    # IMPORTANT:
    # Vega displays only nodes with show_label == 1
    # -------------------------------------------------
    text_mark = config["marks"][-1]

    text_mark["encode"]["update"]["opacity"] = {
        "signal": "datum.show_label == 1 ? 1 : 0"
    }

    # Small gap between node and label
    text_mark["encode"]["update"]["dx"] = {
        "value": 7
    }

    text_mark["encode"]["update"]["dy"] = {
        "value": 0
    }

    text_mark["encode"]["update"]["align"] = {
        "value": "left"
    }

    text_mark["encode"]["update"]["baseline"] = {
        "value": "middle"
    }

    # -------------------------------------------------
    # Save Vega JSON
    # -------------------------------------------------
    output_file = (
        Path(output_path)
        / f"{fn}_vega_viz_config.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf8"
    ) as outfile:

        json.dump(
            config,
            outfile,
            indent=4
        )

    print(
        f'File saved as "{output_file}"'
    )

    return config

In [3448]:
#print(output_path)
#import os

#print(os.getcwd())
#print(os.path.exists("./vega_config/vega_config.json"))
#to show the root_id for each organ

truncated_nodes[truncated_nodes["name"].str.lower().eq("brain")][["graph_int_id", "name", "type", "organ", "ontology_id"]]


,graph_int_id,name,type,organ,ontology_id
4792,5605.0,brain,AS,allen_brain,UBERON:0000955


In [3449]:
ORGAN_FTU_NAMES = {
    "liver": {
        "liver lobule",
    },

    "kidney": {
        "cortical collecting duct",
        "descending limb of loop of henle",
        "inner medullary collecting duct",
        "loop of henle ascending limb thin segment",
        "nephron",
        "outer medullary collecting duct",
        "renal corpuscle",
        "thick ascending limb of loop of henle",
    },

    "large_intestine": {
        "crypt of lieberkuhn",
        "crypt of lieberkuhn of large intestine",
    },

    "small_intestine": {
        "intestinal villus",
    },

    "mouth": {
        "gingival attachment",
        "parotid gland ducto-acinar unit",
        "sublingual gland ducto-acinar unit",
        "submandibular gland ducto-acinar unit",
    },

    "prostate": {
        "prostate glandular acinus",
    },

    "lung": {
        "alveolus of lung",
        "bronchus submucosal gland",
    },

    "skin": {
        "dermal papilla",
        "epidermal ridge of digit",
        "hair follicle",
    },

    "brain": set(),
    "heart": set(),
}

In [3450]:
'''brain_root = 5605   # your current root ID

brain_nodes = truncated_nodes[
    (truncated_nodes["organ"] == "brain") |
    (truncated_nodes["graph_int_id"] == brain_root)
].copy()

brain_edges = truncated_edges[
    (truncated_edges["organ"] == "brain") |
    (truncated_edges["source_int"] == brain_root) |
    (truncated_edges["target_int"] == brain_root)
].copy()

print("Brain nodes:", len(brain_nodes))
print("Brain edges:", len(brain_edges))'''
#truncated_nodes[truncated_nodes["graph_int_id"] == 5605]
'''truncated_nodes[
    truncated_nodes["name"].str.contains("brain", case=False, na=False)
][["graph_int_id", "name", "organ", "type"]]'''

'truncated_nodes[\n    truncated_nodes["name"].str.contains("brain", case=False, na=False)\n][["graph_int_id", "name", "organ", "type"]]'

In [3451]:
from pathlib import Path

PROJECT_DIR = Path(
    "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update"
)

output_path = str(PROJECT_DIR)

In [3452]:
from pathlib import Path

vega_template = (
    Path(output_path)
    / "vega_config"
    / "vega_config_horizontal.json"
)

print("Template:", vega_template)
print("Exists:", vega_template.exists())

Template: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/vega_config/vega_config_horizontal.json
Exists: False


In [3453]:
from pathlib import Path

BUTTERFLY_DIR = Path(
    "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update"
)

matches = list(
    BUTTERFLY_DIR.rglob("vega_config_horizontal.json")
)

print("Found:", len(matches))

for path in matches:
    print(path)

Found: 2
/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftus_connected_by_vasculature_per_organ/vega_config_horizontal.json
/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/vega_config/vega_config_horizontal.json


In [3454]:
vega_template = (
    Path(output_path)
    / "vega_config"
    / "vega_config_horizontal.json"
)

print("Template:", vega_template)
print("Exists:", vega_template.exists())

Template: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/vega_config/vega_config_horizontal.json
Exists: False


In [3455]:
print("CURRENT output_path:")
print(output_path)

CURRENT output_path:
/Users/zeynepamac/Desktop/Butterfly_v2.5_Update


In [3456]:
from pathlib import Path

output_path = str(
    Path(
        "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator"
    )
)

print("FINAL output_path =", output_path)

FINAL output_path = /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator


In [3457]:
'''vega_config_heart = construct_network_create_vega_viz(
    heart_nodes,
    heart_edges,
    scenegraph=True,
    show_labels=True,
    filename="heart_tree",
    root_id=380
)
create_vega_viz(vega_config_heart, "heart_tree", scenegraph=False)'''

'vega_config_heart = construct_network_create_vega_viz(\n    heart_nodes,\n    heart_edges,\n    scenegraph=True,\n    show_labels=True,\n    filename="heart_tree",\n    root_id=380\n)\ncreate_vega_viz(vega_config_heart, "heart_tree", scenegraph=False)'

In [3458]:
vega_config_female_with_ids = construct_network_create_vega_viz(truncated_nodes, truncated_edges, only_female=True, scenegraph=True, show_labels=True, filename='butterfly_ids') #id shown - this is used for the coordinates 
vega_config_female = construct_network_create_vega_viz(truncated_nodes, truncated_edges, only_female=True, scenegraph=True, show_labels=False) #no name or id shown, based on id (scenegraph) - this is used for the viz

ValueError: cannot convert float NaN to integer

In [ ]:
vega_config_male_with_ids = construct_network_create_vega_viz(truncated_nodes, truncated_edges, only_male=True, scenegraph=True, show_labels=True, filename='butterfly_ids')
vega_config_male = construct_network_create_vega_viz(truncated_nodes, truncated_edges, only_male=True, scenegraph=True)

File saved as "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/butterfly_ids_male_vega_viz_config.json"
File saved as "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/butterfly_male_vega_viz_config.json"


In [ ]:
vega_config_female_with_names = construct_network_create_vega_viz(truncated_nodes, truncated_edges, only_female=True, scenegraph=False, show_labels=True, filename='butterfly_names') 
vega_config_female_with_names = update_vega_config(vega_config_female_with_names, 'female')

vega_config_male_with_names = construct_network_create_vega_viz(truncated_nodes, truncated_edges, only_male=True, scenegraph=False, show_labels=True, filename='butterfly_names') 
vega_config_male_with_names = update_vega_config(vega_config_male_with_names, 'male')

File saved as "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/butterfly_names_female_vega_viz_config.json"
File saved as "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/butterfly_names_female_vega_viz_updated.json"
File saved as "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/butterfly_names_male_vega_viz_config.json"
File saved as "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/butterfly_names_male_vega_viz_updated.json"


In [ ]:
def create_vega_viz(config, filename, scenegraph=False):
    '''
    Create the visualization based on the vega config file. The visualization can be saved as SVG or as a scenegraph JSON file.

    Parameters:
    config: dict
        The vega config file.
    filename: str
        The name of the file to be saved.
    scenegraph: bool
        If True, the scenegraph JSON file will be saved, otherwise the SVG file will be saved.
    '''

    if scenegraph:
        scenegraph_data = vlc.vega_to_scenegraph(vg_spec=config)

        with open(f"{output_path}/vega_{filename}_scenegraph.json", "w") as outfile:
            outfile.write(json.dumps(scenegraph_data, indent=4))

        return scenegraph_data
    
    else:
        svg_str = vlc.vega_to_svg(vg_spec=config,)

        with open(f"{output_path}/vega_{filename}_viz.svg", "wt") as f:
            f.write(svg_str)

        return svg_str

In [3459]:
# ============================================================
# FINAL ORGAN-SPECIFIC TREE GENERATION
# ============================================================

# ------------------------------------------------------------
# Requested organs
# ------------------------------------------------------------

organ_specs = [
    {"organ": "brain",           "root_id": 5605},
    {"organ": "heart",           "root_id": 380},
    {"organ": "large_intestine", "root_id": 7661},
    {"organ": "small_intestine", "root_id": 7473},
    {"organ": "liver",           "root_id": 7246},
    {"organ": "prostate",        "root_id": 9201},
    {"organ": "kidney",          "root_id": 8991},
    {"organ": "mouth",           "root_id": 277},
    {"organ": "lung",            "root_id": 164},
    {"organ": "skin",            "root_id": 6719},
]

# Exact baseline fingerprints from the current validated organ trees.
# Each digest covers graph ID, parent edge, name, node type, ontology ID,
# color, and organ. Layout-only edits must leave every digest unchanged.
EXPECTED_TREE_TOPOLOGY_SHA256 = {
    "lung": "9678ee3adc7e682be87de9effba88920134d55a8a40d076a0832aa6e1bcb5a22",
    "prostate": "6fa866e8ce595f3e915589468e506ff7acfc519438e7216169457035b9421510",
    "skin": "b1c89cbe6dbec89ae300a763e5d88acef82be3ec1eca8fc73e658ac0f99ef675",
    "small_intestine": "1f7f538c03186989e752fc3f55ed657dea006b0bb704d6c52432b583418656b0",
    "large_intestine": "739c36a506952ee65903154d12b0e3ee35e68e33e468c98f41bdde4a927d9d6e",
    "liver": "6c049dd1e85687e4e9f44b428028fa49c4f04913e736953ed061a0703df00dc4",
    "kidney": "f2bc8a3578017f752205e8a54b2b0e105419f314565f78865868c2ac55570b45",
    "brain": "9f2f0435cba30bfcc9d63006c725409727d60cbfdf4cec312c519f646ea8490b",
    "heart": "356e95df2ad10a854031200200332874b14056e231f1fd3c6fa122df25e756c9",
    "mouth": "46e9a9aa6526da8a1de68d445ea1d86ca595a3975f70129c63b25a9c61c3a9e3",
}


# ------------------------------------------------------------
# Table_S1-derived FTUs to display for each requested organ
#
# Table_S1 provides FTU names and ontology IDs but no organ column.
# Organ ownership comes from the curated ORGAN_FTU_NAMES definitions;
# PathVessel and the node organ metadata column are not used.
# ------------------------------------------------------------

ORGAN_FTU_IDS = {}

for organ_spec in organ_specs:
    organ_name = organ_spec["organ"]
    resolved_ids = set()

    for configured_ftu_name in ORGAN_FTU_NAMES.get(
        organ_name,
        set()
    ):
        normalized_ftu_name = (
            str(configured_ftu_name)
            .strip()
            .lower()
        )

        table_s1_ftu_id = TABLE_S1_FTU_ID_BY_NAME.get(
            normalized_ftu_name
        )

        if table_s1_ftu_id is None:
            continue

        resolved_ids.add(
            TABLE_S1_GRAPH_FTU_ID_OVERRIDES.get(
                normalized_ftu_name,
                table_s1_ftu_id
            )
        )

    ORGAN_FTU_IDS[organ_name] = resolved_ids

# Brain and Heart are explicitly non-FTU for the current Table_S1
# inventory. Do not infer or invent FTUs for them.
assert ORGAN_FTU_IDS["brain"] == set()
assert ORGAN_FTU_IDS["heart"] == set()


# Preferred display text for FTUs whose source name is blank or broader.
FTU_LABEL_OVERRIDES = {
    "UBERON:8000011": "Parotid gland ducto-acinar unit",
    "UBERON:8000012": "Sublingual gland ducto-acinar unit",
    "UBERON:8000013": "Submandibular gland ducto-acinar unit",
}


# ------------------------------------------------------------
# Build the full directed graph ONCE
# ------------------------------------------------------------

graph_edges = truncated_edges.dropna(
    subset=["source_int", "target_int"]
).copy()

graph_edges["source_int"] = pd.to_numeric(
    graph_edges["source_int"],
    errors="coerce"
)

graph_edges["target_int"] = pd.to_numeric(
    graph_edges["target_int"],
    errors="coerce"
)

graph_edges = graph_edges.dropna(
    subset=["source_int", "target_int"]
).copy()

graph_edges["source_int"] = (
    graph_edges["source_int"].astype(int)
)

graph_edges["target_int"] = (
    graph_edges["target_int"].astype(int)
)

full_graph = nx.from_pandas_edgelist(
    graph_edges,
    source="source_int",
    target="target_int",
    create_using=nx.DiGraph()
)

body_root_id = 0


# ------------------------------------------------------------
# Graph-node metadata lookups
#
# These use graph IDs only. The organ metadata column is not
# used to determine subtree membership.
# ------------------------------------------------------------

node_metadata = truncated_nodes.copy()

node_metadata["graph_int_id"] = pd.to_numeric(
    node_metadata["graph_int_id"],
    errors="coerce"
)

node_metadata = (
    node_metadata
    .dropna(subset=["graph_int_id"])
    .copy()
)

node_metadata["graph_int_id"] = (
    node_metadata["graph_int_id"].astype(int)
)

node_metadata = (
    node_metadata
    .drop_duplicates(subset=["graph_int_id"], keep="first")
    .copy()
)

node_type_by_id = dict(
    zip(
        node_metadata["graph_int_id"],
        node_metadata["type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )
)

node_ontology_by_id = dict(
    zip(
        node_metadata["graph_int_id"],
        node_metadata["ontology_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
)

global_ftu_ids = {
    str(value).strip()
    for value in CONFIRMED_GRAPH_FTU_IDS
}


# ============================================================
# GENERATE EACH ORGAN
# ============================================================

for spec in organ_specs:

    organ = spec["organ"]
    organ_root_id = int(spec["root_id"])

    print("\n" + "=" * 75)
    print("BUILDING:", organ.upper())
    print("ROOT ID:", organ_root_id)
    print("=" * 75)

    # --------------------------------------------------------
    # Check organ root
    # --------------------------------------------------------

    if organ_root_id not in full_graph:

        print(
            f"SKIPPING {organ}: "
            f"root {organ_root_id} not found"
        )

        continue

    # --------------------------------------------------------
    # Body -> Organ path
    # --------------------------------------------------------

    try:

        body_to_organ = nx.shortest_path(
            full_graph,
            source=body_root_id,
            target=organ_root_id
        )

    except nx.NetworkXNoPath:

        print(
            f"SKIPPING {organ}: "
            "no path from Body to organ"
        )

        continue

    # --------------------------------------------------------
    # All nodes underneath this organ
    # --------------------------------------------------------

    organ_descendants = nx.descendants(
        full_graph,
        organ_root_id
    )

    organ_reachable_ids = (
        set(organ_descendants)
        | {organ_root_id}
    )

    # --------------------------------------------------------
    # FTUs requested for this organ
    # --------------------------------------------------------

    desired_ftu_ids = {
        str(x).strip()
        for x in ORGAN_FTU_IDS.get(
            organ,
            set()
        )
    }

    print(
        "Requested FTU IDs:",
        desired_ftu_ids
    )

    # --------------------------------------------------------
    # Find ONE graph node for each FTU ID
    # --------------------------------------------------------

    representative_ftu_nodes = {}

    for ftu_id in sorted(desired_ftu_ids):

        candidates_df = truncated_nodes[
            truncated_nodes["ontology_id"]
            .astype(str)
            .str.strip()
            .eq(ftu_id)
        ].copy()

        # ----------------------------------------------------
        # Safe numeric graph IDs
        # ----------------------------------------------------

        candidates_df["graph_int_id"] = pd.to_numeric(
            candidates_df["graph_int_id"],
            errors="coerce"
        )

        candidates_df = candidates_df.dropna(
            subset=["graph_int_id"]
        ).copy()

        candidates_df["graph_int_id"] = (
            candidates_df["graph_int_id"]
            .astype(int)
        )

        # ----------------------------------------------------
        # Remove duplicate dataframe rows
        # ----------------------------------------------------

        candidates_df = (
            candidates_df
            .drop_duplicates(
                subset=["graph_int_id"],
                keep="first"
            )
            .sort_values("graph_int_id")
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Only candidates actually below this organ
        # ----------------------------------------------------

        candidates_df = candidates_df[
            candidates_df["graph_int_id"]
            .isin(organ_reachable_ids)
        ].copy()

        # ----------------------------------------------------
        # For Lung, explicitly use the alveolus name
        # ----------------------------------------------------

        if (
            organ == "lung"
            and ftu_id == "UBERON:0002299"
        ):

            named_candidates = candidates_df[
                candidates_df["name"]
                .astype(str)
                .str.strip()
                .str.lower()
                .eq("alveolus of lung")
            ].copy()

            if len(named_candidates) > 0:
                candidates_df = named_candidates

        # ----------------------------------------------------
        # Pick the FIRST reachable candidate
        # ----------------------------------------------------

        candidates = (
            candidates_df["graph_int_id"]
            .astype(int)
            .tolist()
        )

        for candidate in candidates:

            candidate = int(candidate)

            if candidate not in full_graph:
                continue

            try:
                nx.shortest_path(
                    full_graph,
                    source=organ_root_id,
                    target=candidate
                )

                representative_ftu_nodes[
                    ftu_id
                ] = candidate

                break

            except nx.NetworkXNoPath:
                continue

    print(
        "Representative FTUs:",
        representative_ftu_nodes
    )

    missing_ftu_ids = (
        desired_ftu_ids
        - set(representative_ftu_nodes)
    )

    if missing_ftu_ids:
        print(
            "WARNING - requested FTUs not reachable from",
            organ_root_id,
            ":",
            sorted(missing_ftu_ids)
        )

    # --------------------------------------------------------
    # Build ONLY the selected tree
    # --------------------------------------------------------

    selected_node_ids = {
        int(x)
        for x in body_to_organ
    }

    selected_node_ids.add(
        organ_root_id
    )

    # --------------------------------------------------------
    # FTU organs:
    #   organ -> one representative FTU -> CT/other layer
    #
    # Non-FTU organs:
    #   organ -> first-level AS -> second-level AS -> direct
    #   CT/other children. This is a controlled structural
    #   view, not the unrestricted full descendant subtree.
    # --------------------------------------------------------

    first_level_as_ids = set()
    second_level_as_ids = set()
    selected_structural_ct_ids = set()

    if desired_ftu_ids:

        for ftu_id, ftu_node in (
            representative_ftu_nodes.items()
        ):

            path_to_ftu = nx.shortest_path(
                full_graph,
                source=organ_root_id,
                target=ftu_node
            )

            selected_node_ids.update(
                int(x)
                for x in path_to_ftu
            )

            # Traverse the representative FTU branch using
            # source edges. Stop at any additional FTU node so
            # duplicate FTU occurrences and their branches are
            # not reintroduced below the chosen representative.
            descendant_stack = [
                int(node_id)
                for node_id in full_graph.successors(
                    ftu_node
                )
            ]

            visited_descendants = set()

            while descendant_stack:
                descendant_id = int(
                    descendant_stack.pop()
                )

                if descendant_id in visited_descendants:
                    continue

                visited_descendants.add(
                    descendant_id
                )

                if node_ontology_by_id.get(
                    descendant_id,
                    ""
                ) in global_ftu_ids:
                    continue

                selected_node_ids.add(
                    descendant_id
                )

                descendant_stack.extend(
                    int(node_id)
                    for node_id in full_graph.successors(
                        descendant_id
                    )
                )

    else:

        # ----------------------------------------------------
        # Candidate AS hierarchy diagnostic BEFORE pruning.
        #
        # Print every AS candidate grouped by directed depth
        # from the organ root. The retained levels below are
        # therefore explicit and graph-derived, never selected
        # by dataframe order, head(n), or visual appearance.
        # ----------------------------------------------------

        organ_depth_by_id = (
            nx.single_source_shortest_path_length(
                full_graph,
                organ_root_id
            )
        )

        candidate_as_ids = {
            int(node_id)
            for node_id, depth in organ_depth_by_id.items()
            if node_type_by_id.get(
                int(node_id),
                ""
            ) == "AS"
        }

        print(
            "\nCANDIDATE AS HIERARCHY BEFORE PRUNING:",
            organ
        )

        for candidate_depth in sorted(
            {
                int(organ_depth_by_id[node_id])
                for node_id in candidate_as_ids
            }
        ):
            print("  DEPTH", candidate_depth)

            for candidate_as_id in sorted(
                node_id
                for node_id in candidate_as_ids
                if int(
                    organ_depth_by_id[node_id]
                ) == candidate_depth
            ):
                metadata_rows = node_metadata[
                    node_metadata["graph_int_id"]
                    == candidate_as_id
                ]

                if len(metadata_rows) > 0:
                    metadata_row = metadata_rows.iloc[0]
                    candidate_name = str(
                        metadata_row["name"]
                    )
                    candidate_ontology_id = str(
                        metadata_row["ontology_id"]
                    )
                else:
                    candidate_name = ""
                    candidate_ontology_id = ""

                direct_as_children = sum(
                    node_type_by_id.get(
                        int(child_id),
                        ""
                    ) == "AS"
                    for child_id in full_graph.successors(
                        candidate_as_id
                    )
                )

                direct_ct_children = sum(
                    node_type_by_id.get(
                        int(child_id),
                        ""
                    ) != "AS"
                    and node_ontology_by_id.get(
                        int(child_id),
                        ""
                    ) not in global_ftu_ids
                    for child_id in full_graph.successors(
                        candidate_as_id
                    )
                )

                print(
                    {
                        "graph_int_id": candidate_as_id,
                        "name": candidate_name,
                        "ontology_id": candidate_ontology_id,
                        "depth": candidate_depth,
                        "direct_as_children": int(
                            direct_as_children
                        ),
                        "direct_ct_children": int(
                            direct_ct_children
                        ),
                    }
                )

        print(
            "RETAINED NON-FTU STRUCTURAL LEVELS:",
            "AS depth 1 and depth 2, plus their direct",
            "non-AS/non-FTU children"
        )

        first_level_as_ids = {
            int(node_id)
            for node_id in full_graph.successors(
                organ_root_id
            )
            if node_type_by_id.get(
                int(node_id),
                ""
            ) == "AS"
        }

        second_level_as_ids = {
            int(node_id)
            for first_level_id in first_level_as_ids
            for node_id in full_graph.successors(
                first_level_id
            )
            if node_type_by_id.get(
                int(node_id),
                ""
            ) == "AS"
        }

        selected_as_ids = (
            first_level_as_ids
            | second_level_as_ids
        )

        selected_node_ids.update(
            selected_as_ids
        )

        selected_structural_ct_ids = {
            int(node_id)
            for as_node_id in selected_as_ids
            for node_id in full_graph.successors(
                as_node_id
            )
            if node_type_by_id.get(
                int(node_id),
                ""
            ) != "AS"
            and node_ontology_by_id.get(
                int(node_id),
                ""
            ) not in global_ftu_ids
        }

        selected_node_ids.update(
            selected_structural_ct_ids
        )

    # --------------------------------------------------------
    # Extract selected nodes
    # --------------------------------------------------------

    node_graph_ids = pd.to_numeric(
        truncated_nodes["graph_int_id"],
        errors="coerce"
    )

    organ_nodes = truncated_nodes[
        node_graph_ids.isin(
            selected_node_ids
        )
    ].copy()

    # --------------------------------------------------------
    # Exactly ONE dataframe row per graph node
    # --------------------------------------------------------

    organ_nodes["graph_int_id"] = pd.to_numeric(
        organ_nodes["graph_int_id"],
        errors="coerce"
    )

    organ_nodes = organ_nodes.dropna(
        subset=["graph_int_id"]
    ).copy()

    organ_nodes["graph_int_id"] = (
        organ_nodes["graph_int_id"]
        .astype(int)
    )

    organ_nodes = (
        organ_nodes
        .drop_duplicates(
            subset=["graph_int_id"],
            keep="first"
        )
        .copy()
    )

    # --------------------------------------------------------
    # Extract selected edges
    # --------------------------------------------------------

    source_ids = pd.to_numeric(
        truncated_edges["source_int"],
        errors="coerce"
    )

    target_ids = pd.to_numeric(
        truncated_edges["target_int"],
        errors="coerce"
    )

    organ_edges = truncated_edges[
        source_ids.isin(selected_node_ids)
        &
        target_ids.isin(selected_node_ids)
    ].copy()

    # Remove duplicate edges

    organ_edges = (
        organ_edges
        .drop_duplicates(
            subset=[
                "source_int",
                "target_int"
            ],
            keep="first"
        )
        .copy()
    )

    # --------------------------------------------------------
    # Labels
    #
    # Label every node on the directed Body -> organ -> displayed FTU
    # paths. Nodes outside those paths remain intentionally unlabeled.
    # --------------------------------------------------------

    label_paths = [
        [int(node_id) for node_id in body_to_organ]
    ]

    for ftu_node_id in sorted(
        representative_ftu_nodes.values()
    ):
        label_paths.append(
            [
                int(node_id)
                for node_id in nx.shortest_path(
                    full_graph,
                    source=body_root_id,
                    target=int(ftu_node_id)
                )
            ]
        )

    label_ids = {
        node_id
        for label_path in label_paths
        for node_id in label_path
    }

    path_label_depth = {
        node_id: nx.shortest_path_length(
            full_graph,
            source=body_root_id,
            target=node_id
        )
        for node_id in label_ids
    }

    organ_nodes["show_label"] = (
        organ_nodes["graph_int_id"]
        .isin(label_ids)
        .astype(int)
    )

    # --------------------------------------------------------
    # Validation report (printed before Vega/SVG rendering)
    # --------------------------------------------------------

    selected_ontology_ids = (
        organ_nodes["ontology_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    selected_types = (
        organ_nodes["type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    selected_ftu_mask = (
        selected_ontology_ids.isin(global_ftu_ids)
    )

    selected_as_mask = (
        selected_types.eq("AS")
        & ~selected_ftu_mask
    )

    selected_ct_other_mask = ~(
        selected_ftu_mask
        | selected_as_mask
    )

    selected_ftu_report = []

    for ftu_id, graph_id in sorted(
        representative_ftu_nodes.items()
    ):
        selected_ftu_report.append(
            {
                "ontology_id": ftu_id,
                "graph_id": int(graph_id),
            }
        )

    body_to_organ_report = [
        int(node_id)
        for node_id in body_to_organ
    ]

    validation_report = {
        "organ": organ,
        "root_id": organ_root_id,
        "body_to_organ_path": body_to_organ_report,
        "as_nodes": int(selected_as_mask.sum()),
        "ct_other_nodes": int(
            selected_ct_other_mask.sum()
        ),
        "selected_nodes": len(organ_nodes),
        "selected_edges": len(organ_edges),
        "ftu_nodes": int(selected_ftu_mask.sum()),
        "unique_ftu_ontology_ids": int(
            selected_ontology_ids[
                selected_ftu_mask
            ].nunique()
        ),
        "selected_ftus": selected_ftu_report,
    }

    print("\nVALIDATION REPORT:")

    for key, value in validation_report.items():
        print(f"  {key}: {value}")

    print(
        "VALIDATION SUMMARY | "
        "organ | selected nodes | AS | FTU | CT/other | "
        "unique FTU IDs"
    )

    print(
        "VALIDATION SUMMARY |",
        organ,
        "|",
        len(organ_nodes),
        "|",
        int(selected_as_mask.sum()),
        "|",
        int(selected_ftu_mask.sum()),
        "|",
        int(selected_ct_other_mask.sum()),
        "|",
        int(
            selected_ontology_ids[
                selected_ftu_mask
            ].nunique()
        )
    )

    selected_graph = full_graph.subgraph(
        selected_node_ids
    )

    # --------------------------------------------------------
    # Per-representative FTU validation
    # --------------------------------------------------------

    for ftu_id, graph_id in sorted(
        representative_ftu_nodes.items()
    ):
        graph_id = int(graph_id)

        representative_rows = organ_nodes[
            organ_nodes["graph_int_id"] == graph_id
        ]

        source_name = (
            str(representative_rows.iloc[0]["name"])
            .strip()
            if len(representative_rows) > 0
            else ""
        )

        ftu_name = FTU_LABEL_OVERRIDES.get(
            ftu_id,
            source_name
        )

        representative_path = nx.shortest_path(
            full_graph,
            source=organ_root_id,
            target=graph_id
        )

        representative_descendants = nx.descendants(
            selected_graph,
            graph_id
        )

        representative_ct_descendants = sum(
            node_type_by_id.get(
                int(node_id),
                ""
            ) != "AS"
            and node_ontology_by_id.get(
                int(node_id),
                ""
            ) not in global_ftu_ids
            for node_id in representative_descendants
        )

        print(
            "FTU VALIDATION |",
            ftu_name,
            "|",
            "| Table_S1 ID:",
            GRAPH_FTU_ID_TO_TABLE_S1_ID.get(
                ftu_id,
                ftu_id
            ),
            "| graph FTU ID:",
            ftu_id,
            "| representative graph ID:",
            graph_id,
            "| path length:",
            len(representative_path) - 1,
            "| CT descendants:",
            int(representative_ct_descendants)
        )

    # --------------------------------------------------------
    # Lung: show every configured FTU candidate and whether it
    # is actually reachable from Lung root 164.
    # --------------------------------------------------------

    if organ == "lung":

        print("\nLUNG FTU CANDIDATE VALIDATION:")

        for ftu_id in sorted(desired_ftu_ids):

            all_candidate_rows = node_metadata[
                node_metadata["ontology_id"]
                .fillna("")
                .astype(str)
                .str.strip()
                .eq(ftu_id)
            ].sort_values("graph_int_id")

            all_candidate_ids = [
                int(value)
                for value in all_candidate_rows[
                    "graph_int_id"
                ].tolist()
            ]

            reachable_candidate_ids = [
                graph_id
                for graph_id in all_candidate_ids
                if graph_id in full_graph
                and nx.has_path(
                    full_graph,
                    organ_root_id,
                    graph_id
                )
            ]

            selected_representative = (
                representative_ftu_nodes.get(ftu_id)
            )

            if selected_representative is not None:
                selected_representative = int(
                    selected_representative
                )

                representative_path = nx.shortest_path(
                    full_graph,
                    source=organ_root_id,
                    target=selected_representative
                )

                retained_descendants = len(
                    nx.descendants(
                        selected_graph,
                        selected_representative
                    )
                )
            else:
                representative_path = None
                retained_descendants = 0

            source_name = (
                str(all_candidate_rows.iloc[0]["name"])
                .strip()
                if len(all_candidate_rows) > 0
                else ""
            )

            print(
                {
                    "ftu_name": FTU_LABEL_OVERRIDES.get(
                        ftu_id,
                        source_name
                    ),
                    "ontology_id": ftu_id,
                    "all_candidate_graph_ids": all_candidate_ids,
                    "reachable_from_lung_root_164": (
                        reachable_candidate_ids
                    ),
                    "selected_representative_graph_id": (
                        selected_representative
                    ),
                    "path_from_lung_root": representative_path,
                    "retained_descendants": retained_descendants,
                }
            )

    # --------------------------------------------------------
    # Prostate: print the exact Body -> representative FTU path
    # plus the retained orange CT/other descendants.
    # --------------------------------------------------------

    if organ == "prostate":

        print("\nPROSTATE PATH VALIDATION:")

        for graph_id in sorted(
            representative_ftu_nodes.values()
        ):
            prostate_path = nx.shortest_path(
                full_graph,
                source=body_root_id,
                target=int(graph_id)
            )

            for path_node_id in prostate_path:
                path_rows = organ_nodes[
                    organ_nodes["graph_int_id"]
                    == int(path_node_id)
                ]

                if len(path_rows) > 0:
                    path_row = path_rows.iloc[0]

                    print(
                        {
                            "graph_id": int(path_node_id),
                            "name": str(path_row["name"]),
                            "type": str(path_row["type"]),
                            "ontology_id": str(
                                path_row["ontology_id"]
                            ),
                        }
                    )

        print("PROSTATE ORANGE DESCENDANTS:")

        for _, orange_row in organ_nodes[
            selected_ct_other_mask
        ].sort_values("graph_int_id").iterrows():
            print(
                {
                    "graph_id": int(
                        orange_row["graph_int_id"]
                    ),
                    "name": str(orange_row["name"]),
                    "type": str(orange_row["type"]),
                    "ontology_id": str(
                        orange_row["ontology_id"]
                    ),
                }
            )

    # --------------------------------------------------------
    # Brain/Heart: exact deterministic structural rule and one
    # row for every selected AS node, including Body and root.
    # --------------------------------------------------------

    if organ in {"brain", "heart"}:

        print(
            "NON-FTU SELECTION RULE: Body -> organ; direct AS "
            "children of the organ root; direct AS children "
            "of those first-level AS nodes; direct non-AS, "
            "non-FTU children of the selected AS nodes. "
            "Membership and edges use only the directed graph."
        )

        print(
            "  ftu_count_confirmed_zero:",
            int(selected_ftu_mask.sum()) == 0
        )

        selected_id_set = set(
            organ_nodes["graph_int_id"].astype(int)
        )

        selected_as_table = []

        for _, as_row in organ_nodes[
            selected_as_mask
        ].sort_values("graph_int_id").iterrows():

            as_graph_id = int(
                as_row["graph_int_id"]
            )

            if as_graph_id == body_root_id:
                depth_from_organ_root = None
            else:
                depth_from_organ_root = (
                    nx.shortest_path_length(
                        full_graph,
                        source=organ_root_id,
                        target=as_graph_id
                    )
                )

            selected_ct_children = sum(
                int(child_id) in selected_id_set
                and node_type_by_id.get(
                    int(child_id),
                    ""
                ) != "AS"
                and node_ontology_by_id.get(
                    int(child_id),
                    ""
                ) not in global_ftu_ids
                for child_id in full_graph.successors(
                    as_graph_id
                )
            )

            selected_as_table.append(
                {
                    "graph_id": as_graph_id,
                    "name": str(as_row["name"]),
                    "ontology_id": str(
                        as_row["ontology_id"]
                    ),
                    "depth_from_organ_root": (
                        depth_from_organ_root
                    ),
                    "selected_ct_children": int(
                        selected_ct_children
                    ),
                }
            )

        print("SELECTED AS TABLE:")

        for selected_as_row in selected_as_table:
            print(selected_as_row)

        print(
            "  selected_ct_descendants:",
            len(selected_structural_ct_ids)
        )

    # --------------------------------------------------------
    # Build Vega configuration
    # --------------------------------------------------------

    vega_config = construct_network_create_vega_viz(
        organ_nodes,
        organ_edges,
        scenegraph=False,
        show_labels=False,
        filename=f"{organ}_tree",
        root_id=body_root_id
    )

    # Refuse to save if any biological/tree content changed. This digest
    # covers every node identity, parent edge, name, type, ontology ID,
    # color, and organ value; only layout fields are excluded.
    topology_records = sorted(
        (
            int(row["id"]),
            None if "parent" not in row else int(row["parent"]),
            str(row.get("name", "")),
            str(row.get("type", "")),
            str(row.get("ontology_id", "")),
            str(row.get("color", "")),
            str(row.get("organ", "")),
        )
        for row in vega_config["data"][0]["values"]
    )
    topology_digest = hashlib.sha256(
        json.dumps(
            topology_records,
            separators=(",", ":")
        ).encode("utf8")
    ).hexdigest()
    assert topology_digest == EXPECTED_TREE_TOPOLOGY_SHA256[organ], (
        f"{organ}: node/edge/type/color topology changed; refusing to save"
    )

    # ========================================================
    # IMPORTANT:
    # Completely take control of the labels AFTER the
    # constructor returns.
    # ========================================================

    # Map graph ID -> desired label text

    label_text_by_id = {}

    # Body

    label_text_by_id[
        body_root_id
    ] = "Body"

    # Organ root

    organ_root_rows = organ_nodes[
        organ_nodes["graph_int_id"]
        == organ_root_id
    ]

    if len(organ_root_rows) > 0:

        organ_root_name = str(
            organ_root_rows.iloc[0]["name"]
        ).strip()

        label_text_by_id[
            organ_root_id
        ] = organ_root_name.capitalize()

    # FTU representatives

    for ftu_id, node_id in (
        representative_ftu_nodes.items()
    ):

        matching_rows = organ_nodes[
            organ_nodes["graph_int_id"]
            == int(node_id)
        ]

        if len(matching_rows) > 0:

            source_label = str(
                matching_rows.iloc[0]["name"]
            ).strip()

            label_text = FTU_LABEL_OVERRIDES.get(
                ftu_id,
                source_label
            )

            label_text_by_id[
                int(node_id)
            ] = label_text.capitalize()

    # Every intermediate anatomical node on each displayed path. Body,
    # organ, and FTU labels above retain their established wording.
    for _, path_row in organ_nodes[
        organ_nodes["graph_int_id"].isin(label_ids)
    ].iterrows():
        path_node_id = int(path_row["graph_int_id"])
        path_node_name = str(path_row["name"]).strip()
        if not path_node_name or path_node_name.lower() == "nan":
            path_node_name = str(path_row["ontology_id"]).strip()
        label_text_by_id.setdefault(
            path_node_id,
            path_node_name.capitalize()
        )

    # --------------------------------------------------------
    # Put an explicit label_text field into Vega data
    # --------------------------------------------------------

    # Angled labels can remain directly anchored to their own nodes.
    label_dx_by_id = {node_id: 7.0 for node_id in label_ids}

    for row in vega_config["data"][0]["values"]:

        row_id = int(row["id"])

        row["label_text"] = (
            label_text_by_id.get(
                row_id,
                ""
            )
        )

        # Keep show_label consistent

        row["show_label"] = (
            1
            if row_id in label_ids
            else 0
        )

        # All displayed labels share the same baseline below their nodes.
        row["label_dx"] = label_dx_by_id.get(row_id, 8.0)
        row["label_dy"] = -7
        row["label_align"] = "left"
        row["label_baseline"] = "bottom"

    # ========================================================
    # FINAL VEGA TEXT CONTROL
    #
    # This prevents the old constructor from making every
    # FTU occurrence visible.
    # ========================================================

    text_mark = vega_config["marks"][-1]

    # Use our explicit label_text instead of the node name

    text_mark["encode"]["enter"]["text"] = {
        "field": "label_text"
    }

    # Show text only when label_text is non-empty

    text_mark["encode"]["update"]["opacity"] = {
        "signal": "datum.label_text != '' ? 1 : 0"
    }

    # Small gap between node and text

    text_mark["encode"]["update"]["dx"] = {
        "field": "label_dx"
    }

    text_mark["encode"]["update"]["dy"] = {
        "field": "label_dy"
    }

    text_mark["encode"]["update"]["align"] = {
        "field": "label_align"
    }

    text_mark["encode"]["update"]["baseline"] = {
        "field": "label_baseline"
    }

    text_mark["encode"]["update"]["angle"] = {
        "value": -30
    }

    displayed_path_label_ids = {
        int(row["id"])
        for row in vega_config["data"][0]["values"]
        if row.get("label_text", "").strip()
    }
    assert displayed_path_label_ids == label_ids, (
        f"{organ}: displayed labels do not exactly match the requested paths"
    )

    displayed_ftu_labels = [
        row["label_text"].strip().lower()
        for row in vega_config["data"][0]["values"]
        if int(row["id"]) in {
            int(node_id)
            for node_id in representative_ftu_nodes.values()
        }
    ]
    assert len(displayed_ftu_labels) == len(set(displayed_ftu_labels)), (
        f"{organ}: duplicate displayed FTU label detected"
    )
    expected_single_ftu_labels = {
        "small_intestine": "intestinal villus",
        "large_intestine": "crypt of lieberkuhn of large intestine",
        "lung": "alveolus of lung",
    }
    if organ in expected_single_ftu_labels:
        expected_label = expected_single_ftu_labels[organ]
        assert displayed_ftu_labels.count(expected_label) == 1, (
            f"{organ}: expected exactly one displayed {expected_label!r} label"
        )

    # --------------------------------------------------------
    # Save the FINAL patched Vega configuration so the JSON
    # matches the SVG rendered below.
    # --------------------------------------------------------

    final_config_path = (
        Path(output_path)
        / f"{organ}_tree_full_vega_viz_config.json"
    )

    with open(
        final_config_path,
        "w",
        encoding="utf8"
    ) as outfile:
        json.dump(
            vega_config,
            outfile,
            indent=4
        )

    # --------------------------------------------------------
    # Print final labels for debugging
    # --------------------------------------------------------

    print("\nFINAL LABELS:")

    for row in vega_config["data"][0]["values"]:

        if row.get("label_text", "") != "":

            print(
                row["id"],
                "->",
                row["label_text"],
                "|",
                row["ontology_id"]
            )

    # --------------------------------------------------------
    # Render
    # --------------------------------------------------------

    create_vega_viz(
        vega_config,
        f"{organ}_tree",
        scenegraph=False
    )

    print(
        "DONE:",
        organ
    )

output_path = /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator
Table_S1 FTU inventory: 28 unique names / 28 unique ontology IDs
FINAL output_path = /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator

BUILDING: BRAIN
ROOT ID: 5605
Requested FTU IDs: set()
Representative FTUs: {}

CANDIDATE AS HIERARCHY BEFORE PRUNING: brain
  DEPTH 0
{'graph_int_id': 5605, 'name': 'brain', 'ontology_id': 'UBERON:0000955', 'depth': 0, 'direct_as_children': 3, 'direct_ct_children': 23}
  DEPTH 1
{'graph_int_id': 5606, 'name': 'forebrain', 'ontology_id': 'UBERON:0001890', 'depth': 1, 'direct_as_children': 3, 'direct_ct_children': 1}
{'graph_int_id': 6297, 'name': 'midbrain', 'ontology_id': 'UBERON:0001891', 'depth': 1, 'direct_as_children': 3, 'direct_ct_children': 1}
{'graph_int_id': 6331, 'name': 'hindbrain', 'ontology_id': 'UBERON:0002028', 'depth': 1, 'direct_as_children': 3, 'direct_ct_children': 0}
  DEPTH 2
{'graph_int_id': 5607, 'name': 'gray matter of 

In [ ]:
lung_path = nx.shortest_path(
    full_graph,
    source=0,
    target=164
)

print("Lung path IDs:", lung_path)

display(
    truncated_nodes[
        truncated_nodes["graph_int_id"]
        .isin(lung_path)
    ][
        ["graph_int_id", "name", "type", "organ", "ontology_id"]
    ]
    .set_index("graph_int_id")
    .loc[lung_path]
)

Lung path IDs: [0, 66, 67, 68, 164]


,name,type,organ,ontology_id
graph_int_id,,,,
0.0,body,AS,body,UBERON:0013702
66.0,respiratory system,AS,lung,UBERON:0001004
67.0,respiratory tract,AS,lung,UBERON:0000065
68.0,lower respiratory tract,AS,lung,UBERON:0001558
164.0,lung,AS,lung,UBERON:0002048


In [ ]:
%%time
female_svg_names = create_vega_viz(vega_config_female_with_names, 'female_name', scenegraph=False)

CPU times: user 1.11 s, sys: 440 ms, total: 1.54 s
Wall time: 1.04 s


In [ ]:
%%time
male_svg_names = create_vega_viz(vega_config_male_with_names, 'male_name', scenegraph=False)

CPU times: user 551 ms, sys: 245 ms, total: 796 ms
Wall time: 661 ms


In [ ]:
%%time
# female_svg_names = create_vega_viz(vega_config_female, 'female', scenegraph=False)

CPU times: user 3 μs, sys: 4 μs, total: 7 μs
Wall time: 9.06 μs


In [ ]:
%%time
# male_svg_names = create_vega_viz(vega_config_male, 'male', scenegraph=False)

CPU times: user 2 μs, sys: 1e+03 ns, total: 3 μs
Wall time: 4.05 μs


In [ ]:
scenegraph_female = create_vega_viz(vega_config_female_with_ids, 'female_id', scenegraph=True)
scenegraph_male = create_vega_viz(vega_config_male_with_ids, 'male_id', scenegraph=True)

In [ ]:
def get_node_coordinates(scenegraph):
    coords = {}

    def walk(node):
        if isinstance(node, dict):
            if "text" in node and "x" in node and "y" in node:
                coords[node["text"]] = (node["x"], node["y"])

            for key in ["items", "children"]:
                if key in node:
                    for child in node[key]:
                        walk(child)

        elif isinstance(node, list):
            for child in node:
                walk(child)

    walk(scenegraph["scenegraph"])
    return coords

In [ ]:
print(scenegraph_female.keys())
print(scenegraph_female['scenegraph'].keys())
print(len(scenegraph_female['scenegraph']['items']))
print(scenegraph_female['scenegraph']['items'][0].keys())

dict_keys(['width', 'height', 'origin', 'scenegraph'])
dict_keys(['clip', 'interactive', 'items', 'marktype', 'name', 'role', 'zindex'])
1
dict_keys(['items', 'x', 'y', 'width', 'height'])


In [ ]:
coordinates_of_nodes_fem = get_node_coordinates(scenegraph_female)
coordinates_of_nodes_mal = get_node_coordinates(scenegraph_male)

## 3.8 Blood Vasculature 

In [ ]:
df = pd.read_csv(ftu_vessel_url)
print(df.columns.tolist())

['FTU', 'FTUID', 'Vessel', 'VesselID', 'PathStep', 'PathVessel', 'PathVesselID']


In [ ]:
with open(ftu_vessel_url, "r") as f:
    for i in range(5):
        print(repr(f.readline()))

'FTU,FTUID,Vessel,VesselID,PathStep,PathVessel,PathVesselID\n'
'alveolus of lung,UBERON:0002299,alveolar capillary,,-8,right ventricle,UBERON:0002080\n'
'alveolus of lung,UBERON:0002299,alveolar capillary,,-7,pulmonary trunk,UBERON:0002333\n'
'alveolus of lung,UBERON:0002299,alveolar capillary,,-6,left pulmonary artery,UBERON:0001652\n'
'alveolus of lung,UBERON:0002299,alveolar capillary,,-5,segmental pulmonary artery,FMA:8755\n'


In [ ]:
ftu_vessel = pd.read_csv(ftu_vessel_url)[['FTU', 'FTUID', 'Vessel', 'VesselID', 'PathStep', 'PathVessel', 'PathVesselID']]

In [ ]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Load the FTU-vessel data
# ---------------------------------------------------------

ftu_vessel = pd.read_csv("data/Table_S1.csv")

# ---------------------------------------------------------
# 2. Define the FTUs belonging to each requested organ
# ---------------------------------------------------------

ORGAN_FTU_IDS = {
    "brain": set(),

    "heart": set(),

    "large_intestine": {
        "UBERON:0001984",
    },

    "small_intestine": {
        "UBERON:0001213",
    },

    "liver": {
        "UBERON:0004647",
    },

    "prostate": {
        "UBERON:0004179",
    },

    "kidney": {
        "UBERON:0001285",
        "UBERON:0001229",
        "UBERON:0004205",
        "UBERON:0001289",
        "UBERON:0004193",
        "UBERON:0004204",
        "UBERON:0001291",
        "UBERON:0004203",
    },

    "mouth": {
        "UBERON:0001828",
        "UBERON:0001831",
        "UBERON:0001832",
        "UBERON:0001736",
    },

    "lung": {
        "UBERON:0002299",
        "UBERON:8410043",
    },

    "skin": {
        "UBERON:0000412",
        "UBERON:0013487",
        "UBERON:0002073",
    },
}

# ---------------------------------------------------------
# 3. Create output directory
# ---------------------------------------------------------

output_dir = Path("organ_csv")
output_dir.mkdir(exist_ok=True)

# ---------------------------------------------------------
# 4. Assign Type based on PathStep
# ---------------------------------------------------------

def assign_type(path_step):
    if pd.isna(path_step):
        return "Unknown"

    if path_step < 0:
        return "Artery"

    if path_step == 0:
        return "FTU"

    return "Vein"


# ---------------------------------------------------------
# 5. Create one CSV for each organ
# ---------------------------------------------------------

for organ, ftu_list in ORGAN_FTUS.items():

    organ_df = ftu_vessel[
        ftu_vessel["FTU"].isin(ftu_list)
    ].copy()

    # Add organ name
    organ_df["Organ"] = organ

    # Assign artery / FTU / vein
    organ_df["Type"] = organ_df["PathStep"].apply(assign_type)

    # Keep exactly the requested columns
    organ_df = organ_df[
        [
            "FTU",
            "Organ",
            "FTUID",
            "PathStep",
            "PathVessel",
            "PathVesselID",
            "Type",
        ]
    ].copy()

    # Sort the data so each FTU follows its vascular path
    organ_df = organ_df.sort_values(
        ["FTU", "PathStep"]
    ).reset_index(drop=True)

    # Save
    filename = organ.lower().replace(" ", "_")
    output_file = output_dir / f"{filename}_ftu_vascular_data.csv"

    organ_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"{organ}: {len(organ_df)} rows -> {output_file}"
    )

Liver: 87 rows -> organ_csv/liver_ftu_vascular_data.csv
Kidney: 462 rows -> organ_csv/kidney_ftu_vascular_data.csv
L intestine: 34 rows -> organ_csv/l_intestine_ftu_vascular_data.csv
S intestine: 27 rows -> organ_csv/s_intestine_ftu_vascular_data.csv
Mouth: 52 rows -> organ_csv/mouth_ftu_vascular_data.csv
Prostate: 21 rows -> organ_csv/prostate_ftu_vascular_data.csv


In [ ]:
ftu_vessel = pd.read_csv(ftu_vessel_url)
print(ftu_vessel['FTU'].nunique())
print(sorted(ftu_vessel['FTU'].unique()))
print(ftu_vessel.groupby('FTU')['PathVessel'].count().sort_values())

28
['alveolus of lung', 'bronchus submucosal gland', 'chorionic terminal villus', 'cortical collecting duct', 'crypt of Lieberkuhn', 'dermal papilla', 'descending limb of loop of Henle', 'epidermal ridge of digit', 'gingival attachment', 'hair follicle', 'inner medullary collecting duct', 'intercalated duct', 'intestinal villus', 'islet of Langerhans', 'liver lobule', 'loop of Henle ascending limb thin segment', 'nephron', 'outer medullary collecting duct', 'pancreatic acinus', 'parotid gland ducto-acinar unit', 'prostate glandular acinus', 'red pulp of spleen', 'renal corpuscle', 'sublingual gland ducto-acinar unit', 'submandibular gland ducto-acinar unit', 'thick ascending limb of loop of Henle', 'thymus lobule', 'white pulp of spleen']
FTU
thymus lobule                                 14
bronchus submucosal gland                     15
alveolus of lung                              16
sublingual gland ducto-acinar unit            17
parotid gland ducto-acinar unit               17
su

In [ ]:
print(ftu_vessel.columns.tolist())

['FTU', 'FTUID', 'Vessel', 'VesselID', 'PathStep', 'PathVessel', 'PathVesselID']


In [ ]:
def assign_type(path_step):
    if pd.isna(path_step):
        return "Unknown"
    elif path_step < 0:
        return "Artery"
    elif path_step == 0:
        return "FTU"
    else:
        return "Vein"


ftu_vessel["PathStep"] = pd.to_numeric(
    ftu_vessel["PathStep"],
    errors="coerce"
)

ftu_vessel["Type"] = (
    ftu_vessel["PathStep"].apply(assign_type)
)

In [ ]:
unique_pairs = ftu_vessel[
    [
        "FTU",
        "FTUID",
        "PathStep",
        "PathVessel",
        "PathVesselID",
        "Type",
    ]
].drop_duplicates()

display(unique_pairs)

,FTU,FTUID,PathStep,PathVessel,PathVesselID,Type
0,alveolus of lung,UBERON:0002299,-8,right ventricle,UBERON:0002080,Artery
1,alveolus of lung,UBERON:0002299,-7,pulmonary trunk,UBERON:0002333,Artery
2,alveolus of lung,UBERON:0002299,-6,left pulmonary artery,UBERON:0001652,Artery
3,alveolus of lung,UBERON:0002299,-5,segmental pulmonary artery,FMA:8755,Artery
4,alveolus of lung,UBERON:0002299,-4,subsegmental pulmonary artery,UBERON:8600060,Artery
...,...,...,...,...,...,...
1140,white pulp of spleen,UBERON:0001959,-3,lobar artery of spleen,NaN,Artery
1141,white pulp of spleen,UBERON:0001959,-2,segmental branch of lobar artery of spleen,NaN,Artery
1142,white pulp of spleen,UBERON:0001959,-1,spleen trabecular artery,UBERON:0010399,Artery
1143,white pulp of spleen,UBERON:0001959,0,spleen central arteriole,UBERON:0010401,FTU


In [ ]:
## Correct IDs for some PathVessels and FTUs 
ftu_vessel.loc[ftu_vessel['PathVessel'] == 'segmental pulmonary vein', ['PathVesselID']] = 'UBERON:8600051'
ftu_vessel.loc[ftu_vessel['FTU'] == 'intercalated duct', ['FTUID']] = 'UBERON:0014726'
ftu_vessel.loc[ftu_vessel['FTU'] == 'crypt of Lieberkuhn', ['FTUID']] = 'UBERON:0001984'

In [ ]:
def get_node_id(nodes, ontology_id, node_name, node_type):
    # Normalize ID if it's in format like 'fma9283'
    if ontology_id and isinstance(ontology_id, str):
        match = re.match(r'^([a-zA-Z]+):?(\d+)$', ontology_id)
        if match:
            letters = match.group(1).upper()
            numbers = match.group(2)
            ontology_id = f"{letters}:{numbers}"

    # Match ID to node
    ontology_ids = nodes.loc[nodes['ontology_id'] == ontology_id, 'id'].tolist() if ontology_id else []
    if not ontology_ids:
        ontology_ids = nodes.loc[nodes['name'].str.lower() == node_name.lower(), 'id'].tolist()

    if not ontology_ids:
#         print(f'Missing {node_type} Name: {node_name}, {node_type} ID: {ontology_id}')
        ontology_ids = ['None']
        key = node_type+'_'+node_name
        if key not in missing_nodes:
            missing_nodes[key] = ontology_id
        
    return ontology_ids[0]

In [ ]:
missing_nodes = {}
for index, row in ftu_vessel.iterrows():
    ftu_name = row['FTU']
    ftu_id = row['FTUID']
    vessel_name = row['Vessel']
    vessel_id = row['VesselID']
    path_vessel_name = row['PathVessel']
    path_vessel_id = row['PathVesselID']
    
    ftu_ids = get_node_id(nodes, ftu_id, ftu_name, 'FTU')
    ftu_vessel.loc[index,'FTUID_node'] = ftu_ids
    
    vessel_ids = get_node_id(nodes, vessel_id, vessel_name, 'Vessel')
    ftu_vessel.loc[index,'VesselID_node'] = vessel_ids
    
    path_vessel_ids = get_node_id(nodes, path_vessel_id, path_vessel_name, 'PathVessel')
    ftu_vessel.loc[index,'PathVesselID_node'] = path_vessel_ids
    
ftu_vessel = ftu_vessel.replace("None", pd.NA)

In [ ]:
missing_nodes 

{}

In [ ]:
#Add missing pathvessel id nodes
mask = ftu_vessel['PathVesselID_node'].isna() | (ftu_vessel['PathVesselID_node'] == '')

ftu_vessel.loc[mask, 'PathVesselID_node'] = np.where(
    ftu_vessel.loc[mask, 'PathVesselID'].notna() & (ftu_vessel.loc[mask, 'PathVesselID'] != ''),
    ftu_vessel.loc[mask, 'PathVesselID'],
    'ASCTB-TEMP:' + ftu_vessel.loc[mask, 'PathVessel'].astype(str)
)

#Drop empty FTUID_node
ftu_vessel = ftu_vessel.dropna(subset='FTUID_node').reset_index(drop=True)

In [ ]:
ftu_vessel[ftu_vessel['PathVesselID_node'].isnull()]

,FTU,FTUID,Vessel,VesselID,PathStep,PathVessel,PathVesselID,Type,FTUID_node,VesselID_node,PathVesselID_node


In [ ]:
def build_initial_blood_edges(ftu_vessel_df, nodes_df, ftus_to_exclude=None):
    """
    Build initial blood vessel edges from FTU-vessel paths, excluding specified FTUs,
    and explicitly add heart to atria/ventricles connections.

    Parameters:
        ftu_vessel_df (pd.DataFrame): DataFrame with columns ['FTU', 'Vessel', 'PathVesselID_node', 'PathStep', 'FTUID_node', 'VesselID_node'].
        nodes_df (pd.DataFrame): DataFrame with columns ['ontology_id', 'type'].
        ftus_to_exclude (list[str], optional): List of FTU names to exclude (case-insensitive). Default: None.

    Returns:
        pd.DataFrame: DataFrame with columns ['organ', 'source', 'target', 'source_type', 'target_type', 'vessel_type'].
    """

    node_types = nodes_df.set_index('ontology_id')['type'].to_dict()
    new_blood_edges = []

    def make_edge(source, target, organ, vessel_type):
        return {
            'organ': organ,
            'source': source,
            'target': target,
            'source_type': node_types.get(source, 'unknown'),
            'target_type': node_types.get(target, 'unknown'),
            'vessel_type': vessel_type
        }

    exclude_ftus = set(ftu.lower() for ftu in ftus_to_exclude) if ftus_to_exclude else set()

    for (ftu, vessel), group in ftu_vessel_df.groupby(['FTU', 'Vessel']):
        if ftu.lower() in exclude_ftus:
            continue

        row = group.iloc[0]
        organ = 'blood_vasculature'
        path_df = group.dropna(subset=['PathVesselID_node']).sort_values('PathStep')

        arteries = path_df[path_df['PathStep'] <= 0].sort_values('PathStep')
        veins = path_df[path_df['PathStep'] >= 0].sort_values('PathStep', ascending=False)

        art_ids = arteries['PathVesselID_node'].tolist()
        new_blood_edges.extend([
            make_edge(art_ids[i], art_ids[i + 1], organ, 'artery')
            for i in range(len(art_ids) - 1)
        ])

        vein_ids = veins['PathVesselID_node'].tolist()
        new_blood_edges.extend([
            make_edge(vein_ids[i], vein_ids[i + 1], organ, 'vein')
            for i in range(len(vein_ids) - 1)
        ])

        # Vessel to FTU: add both artery and vein edges
        new_blood_edges.append(make_edge(row['VesselID_node'], row['FTUID_node'], organ, 'artery'))
        new_blood_edges.append(make_edge(row['VesselID_node'], row['FTUID_node'], organ, 'vein'))

    # Add heart → ventricle (artery), and heart → atria (vein) edges
    heart_id = 'UBERON:0000948'
    id_of_ventricles = ['UBERON:0002084', 'UBERON:0002080']
    id_of_atria = ['UBERON:0002079', 'UBERON:0002078']

    for ventricle_id in id_of_ventricles:
        new_blood_edges.append(make_edge(heart_id, ventricle_id, 'blood_vasculature', 'artery'))

    for atrium_id in id_of_atria:
        new_blood_edges.append(make_edge(heart_id, atrium_id, 'blood_vasculature', 'vein'))

    return pd.DataFrame(new_blood_edges)

In [ ]:
def augment_edges_with_kept_nodes(edge_df, nodes_df):
    """
    Augments edges by replacing source and target ontology IDs with all kept node IDs.

    Parameters:
        edge_df (pd.DataFrame): DataFrame with at least ['source', 'target'] columns.
        nodes_df (pd.DataFrame): DataFrame with ['ontology_id', 'id', 'keep'] columns.

    Returns:
        pd.DataFrame: Augmented edge DataFrame (original + new combinations).
    """
    # Build lookup: ontology_id → list of kept node IDs
    kept_node_ids = nodes_df[nodes_df['keep'] == True].groupby('ontology_id')['id'].apply(list).to_dict()

    augmented_edges = []

    for _, r in edge_df.iterrows():
        source_ont = r['source']
        target_ont = r['target']

        alt_sources = kept_node_ids.get(source_ont, [])
        alt_targets = kept_node_ids.get(target_ont, [])

        # Alternate sources
        for new_source in alt_sources:
            if new_source != r['source']:
                new_edge = r.copy()
                new_edge['source'] = new_source
                augmented_edges.append(new_edge)

        # Alternate targets
        for new_target in alt_targets:
            if new_target != r['target']:
                new_edge = r.copy()
                new_edge['target'] = new_target
                augmented_edges.append(new_edge)

        # Both source and target replaced
        for new_source in alt_sources:
            for new_target in alt_targets:
                if new_source != r['source'] and new_target != r['target']:
                    new_edge = r.copy()
                    new_edge['source'] = new_source
                    new_edge['target'] = new_target
                    augmented_edges.append(new_edge)

    # Combine and return
    if augmented_edges:
        return pd.concat([edge_df, pd.DataFrame(augmented_edges)], ignore_index=True)
    else:
        return edge_df


In [ ]:
def map_ids_and_coordinates(edge_df, id_map, coord_map, fill_value=1000000):
    """
    Maps internal graph IDs and coordinates to an edge DataFrame.

    Parameters:
        edge_df (pd.DataFrame): DataFrame with 'source' and 'target' columns.
        id_map (dict): Mapping from node ontology ID → internal graph ID (int).
        coord_map (dict): Mapping from internal graph ID (int) → coordinates.
        fill_value (int): Value to use when ID mapping is missing. Default is 1000000.

    Returns:
        pd.DataFrame: Modified edge DataFrame with added 'source_int', 'target_int',
                      'source_coord', 'target_coord' columns.
    """
    df = edge_df.copy()
    
    df['source_int'] = df['source'].map(id_map)
    df['target_int'] = df['target'].map(id_map)
    
    df[['source_int', 'target_int']] = df[['source_int', 'target_int']].fillna(fill_value).astype(int)

    df['source_coord'] = df['source_int'].map(coord_map)
    df['target_coord'] = df['target_int'].map(coord_map)
    
    return df


In [ ]:
def generate_blood_edges(ftu_vessel_df, nodes_df, id_map, coord_map, ftus_to_exclude=None, fill_value=1000000):
    """
    Full pipeline to generate blood edges: initial build, augmentation, and coordinate mapping.

    Parameters:
        ftu_vessel_df (pd.DataFrame): FTU-vessel mapping data.
        nodes_df (pd.DataFrame): Node metadata including 'ontology_id', 'id', 'keep', and 'type'.
        id_map (dict): Mapping from ontology IDs to internal graph node IDs (int).
        coord_map (dict): Mapping from internal node IDs to coordinates.
        ftus_to_exclude (list): Optional list of FTU names (case-insensitive) to exclude. Default is None.
        fill_value (int): Fallback ID if mapping is missing. Default is 1000000.

    Returns:
        pd.DataFrame: Final edge DataFrame with internal IDs and coordinates.
    """
    # Step 1: Build initial blood edges
    initial_edges = build_initial_blood_edges(
        ftu_vessel_df=ftu_vessel_df,
        nodes_df=nodes_df,
        ftus_to_exclude=ftus_to_exclude
    )

    # Step 2: Augment with kept nodes
    augmented_edges = augment_edges_with_kept_nodes(initial_edges, nodes_df)

    # Step 3: Map IDs and coordinates
    final_edges = map_ids_and_coordinates(
        edge_df=augmented_edges,
        id_map=id_map,
        coord_map=coord_map,
        fill_value=fill_value
    )

    return final_edges


In [ ]:
male_only_ftus = ['prostate glandular acinus']
female_only_ftus = None

In [ ]:
female_blood_edges = generate_blood_edges(
    ftu_vessel_df=ftu_vessel,
    nodes_df=nodes,
    id_map=id_to_graph_int_id,
    coord_map=coordinates_of_nodes_fem,
    ftus_to_exclude=male_only_ftus)

female_blood_edges.drop_duplicates(inplace=True)
female_blood_edges.reset_index(drop=True, inplace=True)
female_blood_edges = female_blood_edges.sort_values(by='source_int').reset_index(drop=True)

In [ ]:
male_blood_edges = generate_blood_edges(
    ftu_vessel_df=ftu_vessel,
    nodes_df=nodes,
    id_map=id_to_graph_int_id,
    coord_map=coordinates_of_nodes_mal,
    ftus_to_exclude=female_only_ftus)

male_blood_edges.drop_duplicates(inplace=True)
male_blood_edges.reset_index(drop=True, inplace=True)
male_blood_edges = male_blood_edges.sort_values(by='source_int').reset_index(drop=True)

In [ ]:
#Statistics
print('Female arteries: ', len(female_blood_edges[female_blood_edges['vessel_type'] == 'artery']))
print('Female veins: ', len(female_blood_edges[female_blood_edges['vessel_type'] == 'vein']))
print('Male arteries: ', len(male_blood_edges[male_blood_edges['vessel_type'] == 'artery']))
print('Male veins: ', len(male_blood_edges[male_blood_edges['vessel_type'] == 'vein']))

Female arteries:  343
Female veins:  321
Male arteries:  348
Male veins:  328


In [ ]:
# ============================================================
# DEFINITIVE FTU CHECK FOR BRAIN AND HEART
#
# Tests THREE independent things:
# 1. FTU ontology-ID intersection
# 2. FTU name intersection
# 3. All FTU-like nodes in the organ subtree
# ============================================================

# ------------------------------------------------------------
# Load the complete FTU source
# ------------------------------------------------------------

ftu_source = pd.read_csv("data/Table_S1.csv")

ftu_source["FTU_clean"] = (
    ftu_source["FTU"]
    .astype(str)
    .str.strip()
    .str.lower()
)

ftu_source["FTUID_clean"] = (
    ftu_source["FTUID"]
    .astype(str)
    .str.strip()
)

# One row per actual FTU definition
source_ftus = (
    ftu_source[
        ["FTU_clean", "FTUID_clean"]
    ]
    .drop_duplicates()
    .copy()
)

source_ftu_names = set(
    source_ftus["FTU_clean"]
)

source_ftu_ids = set(
    source_ftus["FTUID_clean"]
)

print("=" * 80)
print("SOURCE FTU INVENTORY")
print("=" * 80)

print("Unique FTU names:", len(source_ftu_names))
print("Unique FTU IDs:", len(source_ftu_ids))

display(
    source_ftus.sort_values(
        ["FTU_clean", "FTUID_clean"]
    )
)

# ------------------------------------------------------------
# Build graph
# ------------------------------------------------------------

full_graph = nx.from_pandas_edgelist(
    truncated_edges.dropna(
        subset=["source_int", "target_int"]
    ),
    source="source_int",
    target="target_int",
    create_using=nx.DiGraph()
)

# ------------------------------------------------------------
# Organ roots
# ------------------------------------------------------------

organ_roots = {
    "brain": 5605,
    "heart": 380,
}

# ------------------------------------------------------------
# Test each organ
# ------------------------------------------------------------

for organ, root_id in organ_roots.items():

    print("\n" + "=" * 80)
    print(organ.upper())
    print("=" * 80)

    # --------------------------------------------------------
    # Get subtree
    # --------------------------------------------------------

    subtree_ids = (
        set(nx.descendants(full_graph, root_id))
        | {root_id}
    )

    node_ids_numeric = pd.to_numeric(
        truncated_nodes["graph_int_id"],
        errors="coerce"
    )

    subtree = truncated_nodes[
        node_ids_numeric.isin(subtree_ids)
    ].copy()

    # --------------------------------------------------------
    # Clean ontology IDs and names
    # --------------------------------------------------------

    subtree["ontology_clean"] = (
        subtree["ontology_id"]
        .astype(str)
        .str.strip()
    )

    subtree["name_clean"] = (
        subtree["name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # --------------------------------------------------------
    # TEST 1:
    # Exact ontology-ID intersection
    # --------------------------------------------------------

    id_matches = subtree[
        subtree["ontology_clean"].isin(
            source_ftu_ids
        )
    ].copy()

    print("\nTEST 1 - FTU ontology IDs in subtree")
    print(
        "Matches:",
        len(id_matches)
    )

    if len(id_matches) > 0:

        display(
            id_matches[
                [
                    "graph_int_id",
                    "name",
                    "ontology_id",
                    "type"
                ]
            ]
            .drop_duplicates(
                subset=[
                    "graph_int_id",
                    "ontology_id"
                ]
            )
            .sort_values(
                "graph_int_id"
            )
        )

    else:
        print("NONE")

    # --------------------------------------------------------
    # TEST 2:
    # Exact FTU-name intersection
    # --------------------------------------------------------

    name_matches = subtree[
        subtree["name_clean"].isin(
            source_ftu_names
        )
    ].copy()

    print("\nTEST 2 - FTU names in subtree")
    print(
        "Matches:",
        len(name_matches)
    )

    if len(name_matches) > 0:

        display(
            name_matches[
                [
                    "graph_int_id",
                    "name",
                    "ontology_id",
                    "type"
                ]
            ]
            .drop_duplicates(
                subset=[
                    "graph_int_id",
                    "name_clean"
                ]
            )
            .sort_values(
                "graph_int_id"
            )
        )

    else:
        print("NONE")

    # --------------------------------------------------------
    # TEST 3:
    # Show all source FTUs that are absent
    # --------------------------------------------------------

    subtree_names = set(
        subtree["name_clean"]
    )

    subtree_ids_clean = set(
        subtree["ontology_clean"]
    )

    missing_by_name = sorted(
        source_ftus[
            ~source_ftus["FTU_clean"].isin(
                subtree_names
            )
        ]["FTU_clean"]
        .unique()
    )

    missing_by_id = sorted(
        source_ftus[
            ~source_ftus["FTUID_clean"].isin(
                subtree_ids_clean
            )
        ]["FTU_clean"]
        .unique()
    )

    print(
        "\nSource FTUs absent by NAME:",
        len(missing_by_name)
    )

    print(
        missing_by_name
    )

    print(
        "\nSource FTUs absent by ID:",
        len(missing_by_id)
    )

    print(
        missing_by_id
    )

    # --------------------------------------------------------
    # Basic subtree stats
    # --------------------------------------------------------

    print("\nSUBTREE SUMMARY")

    print(
        "Total nodes:",
        len(subtree)
    )

    print(
        "AS:",
        (subtree["type"] == "AS").sum()
    )

    print(
        "CT:",
        (subtree["type"] == "CT").sum()
    )

SOURCE FTU INVENTORY
Unique FTU names: 28
Unique FTU IDs: 28


,FTU_clean,FTUID_clean
0,alveolus of lung,UBERON:0002299
16,bronchus submucosal gland,UBERON:8410043
31,chorionic terminal villus,UBERON:8600030
54,cortical collecting duct,UBERON:0004203
78,crypt of lieberkuhn,UBERON:0001983
112,dermal papilla,UBERON:0000412
138,descending limb of loop of henle,UBERON:0001289
164,epidermal ridge of digit,UBERON:0013487
190,gingival attachment,UBERON:8000014
211,hair follicle,UBERON:0002073



BRAIN

TEST 1 - FTU ontology IDs in subtree
Matches: 0
NONE

TEST 2 - FTU names in subtree
Matches: 0
NONE

Source FTUs absent by NAME: 28
['alveolus of lung', 'bronchus submucosal gland', 'chorionic terminal villus', 'cortical collecting duct', 'crypt of lieberkuhn', 'dermal papilla', 'descending limb of loop of henle', 'epidermal ridge of digit', 'gingival attachment', 'hair follicle', 'inner medullary collecting duct', 'intercalated duct', 'intestinal villus', 'islet of langerhans', 'liver lobule', 'loop of henle ascending limb thin segment', 'nephron', 'outer medullary collecting duct', 'pancreatic acinus', 'parotid gland ducto-acinar unit', 'prostate glandular acinus', 'red pulp of spleen', 'renal corpuscle', 'sublingual gland ducto-acinar unit', 'submandibular gland ducto-acinar unit', 'thick ascending limb of loop of henle', 'thymus lobule', 'white pulp of spleen']

Source FTUs absent by ID: 28
['alveolus of lung', 'bronchus submucosal gland', 'chorionic terminal villus', 'cort

## 3.9 Plot Blood Vessels

In [ ]:
'''def generate_vessel_graph(blood_edges, vessel_type):
    # Filter based on vessel type (artery or vein)
    blood_edges_vessel = blood_edges[blood_edges['vessel_type'] == vessel_type]
    
    # Drop unnecessary columns
    blood_edges_vessel = blood_edges_vessel.drop(['organ', 'source_type', 'target_type', 'vessel_type', 'source', 'target'], axis=1)
    
    # Create dictionaries for source and target with non-null coordinates
    fixed_nodes_source = blood_edges_vessel.loc[blood_edges_vessel['source_coord'].notna(), ['source_int', 'source_coord']].set_index('source_int')['source_coord'].to_dict()
    fixed_nodes_target = blood_edges_vessel.loc[blood_edges_vessel['target_coord'].notna(), ['target_int', 'target_coord']].set_index('target_int')['target_coord'].to_dict()
    
    # Combine the two dictionaries
    combined_nodes = {**fixed_nodes_source, **fixed_nodes_target}
    
    # Remove duplicates, keeping the first occurrence
    fixed_pos = {}
    for key, value in combined_nodes.items():
        if value not in fixed_pos.values():
            fixed_pos[key] = value
    
    # Create the graph from the blood edges DataFrame
    G = nx.from_pandas_edgelist(blood_edges_vessel, source='source_int', target='target_int')
    print("Fixed positions:", len(fixed_pos))
    print("First few fixed nodes:", list(fixed_pos.keys())[:10])
    # Calculate the spring layout
    pos = nx.spring_layout(
        G,
        pos=fixed_pos,
        fixed=list(fixed_pos.keys()),
        seed=42,                                # Ensures reproducibility
        iterations=1000,                        # More iterations for layout stability
        dim=2,                                  # 2D layout
        k=0.01 / np.sqrt(len(G))                # Optimal distance scaling based on graph size
    )
    
    return G, pos, list(fixed_pos.keys())'''

'def generate_vessel_graph(blood_edges, vessel_type):\n    # Filter based on vessel type (artery or vein)\n    blood_edges_vessel = blood_edges[blood_edges[\'vessel_type\'] == vessel_type]\n\n    # Drop unnecessary columns\n    blood_edges_vessel = blood_edges_vessel.drop([\'organ\', \'source_type\', \'target_type\', \'vessel_type\', \'source\', \'target\'], axis=1)\n\n    # Create dictionaries for source and target with non-null coordinates\n    fixed_nodes_source = blood_edges_vessel.loc[blood_edges_vessel[\'source_coord\'].notna(), [\'source_int\', \'source_coord\']].set_index(\'source_int\')[\'source_coord\'].to_dict()\n    fixed_nodes_target = blood_edges_vessel.loc[blood_edges_vessel[\'target_coord\'].notna(), [\'target_int\', \'target_coord\']].set_index(\'target_int\')[\'target_coord\'].to_dict()\n\n    # Combine the two dictionaries\n    combined_nodes = {**fixed_nodes_source, **fixed_nodes_target}\n\n    # Remove duplicates, keeping the first occurrence\n    fixed_pos = {}\n 

In [ ]:
# For female arteries
'''G_fem_arteries, artery_pos_female, arteries_fixed_nodes_female = generate_vessel_graph(female_blood_edges, 'artery')

# For female veins
G_fem_veins, vein_pos_female, veins_fixed_nodes_female = generate_vessel_graph(female_blood_edges, 'vein')

# For male arteries
G_male_arteries, artery_pos_male, arteries_fixed_nodes_male = generate_vessel_graph(male_blood_edges, 'artery')

# For male veins
G_male_veins, vein_pos_male, veins_fixed_nodes_male = generate_vessel_graph(male_blood_edges, 'vein')'''

"G_fem_arteries, artery_pos_female, arteries_fixed_nodes_female = generate_vessel_graph(female_blood_edges, 'artery')\n\n# For female veins\nG_fem_veins, vein_pos_female, veins_fixed_nodes_female = generate_vessel_graph(female_blood_edges, 'vein')\n\n# For male arteries\nG_male_arteries, artery_pos_male, arteries_fixed_nodes_male = generate_vessel_graph(male_blood_edges, 'artery')\n\n# For male veins\nG_male_veins, vein_pos_male, veins_fixed_nodes_male = generate_vessel_graph(male_blood_edges, 'vein')"

In [ ]:
'''def plot_blood_vessels_with_bundling(G_arteries, G_veins, vein_pos, artery_pos, filename):

    
    # Create edges and nodes for veins and arteries
    edges = {
        'veins': nx.to_pandas_edgelist(G_veins, source='source_int', target='target_int')[['source_int', 'target_int']].rename(columns={'source_int': 'source', 'target_int': 'target'}),
        'arteries': nx.to_pandas_edgelist(G_arteries, source='source_int', target='target_int')[['source_int', 'target_int']].rename(columns={'source_int': 'source', 'target_int': 'target'})
    }
    
    nodes = {
        'veins': pd.DataFrame.from_dict(vein_pos).T.rename(columns={0: 'x', 1: 'y'}),
        'arteries': pd.DataFrame.from_dict(artery_pos).T.rename(columns={0: 'x', 1: 'y'})
    }
    
    # Apply the hammer bundle method (assumed to be implemented elsewhere)
    hb = {
        'veins': hammer_bundle(nodes['veins'], edges['veins'], initial_bandwidth=.015, decay=0.8, tension=0.99, accuracy=1000),
        'arteries': hammer_bundle(nodes['arteries'], edges['arteries'], initial_bandwidth=.015, decay=0.8, tension=0.99, accuracy=1000)
    }
    
    # Create the plot
    plt.figure(figsize=(canvas_size / 100, canvas_size / 100))
    plt.axes().set_aspect('equal')
    plt.margins(x=0, y=0)
    plt.xlim(0, canvas_size)
    plt.ylim(-canvas_size, 0)
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0, hspace=0, wspace=0)
    plt.tight_layout(pad=0, h_pad=0, w_pad=0, rect=(0, 0, 1, 1))
    plt.axis('off')

    # Plot the veins and arteries
    plt.plot(hb['veins']['x'], -hb['veins']['y'], color='tab:blue', alpha=0.8)
    plt.plot(hb['arteries']['x'], -hb['arteries']['y'], color='tab:red', alpha=0.8)

    # Adjust plot layout and margins
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.tight_layout(pad=0, h_pad=0, w_pad=0, rect=(0, 0, 1, 1))
    plt.axis('off')
    plt.gca().set_axis_off()
    plt.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)
    plt.margins(0, 0)
    plt.gca().xaxis.set_major_locator(plt.NullLocator())
    plt.gca().yaxis.set_major_locator(plt.NullLocator())

    # Save the figure
    plt.savefig(f'{output_path}/blood_viz_{filename}_bundled.svg', pad_inches=0.0, transparent=True, bbox_inches=0)
    print(f'Fig saved as "{output_path}/blood_viz_{filename}_bundled.svg"')
    plt.show()'''


'def plot_blood_vessels_with_bundling(G_arteries, G_veins, vein_pos, artery_pos, filename):\n\n\n    # Create edges and nodes for veins and arteries\n    edges = {\n        \'veins\': nx.to_pandas_edgelist(G_veins, source=\'source_int\', target=\'target_int\')[[\'source_int\', \'target_int\']].rename(columns={\'source_int\': \'source\', \'target_int\': \'target\'}),\n        \'arteries\': nx.to_pandas_edgelist(G_arteries, source=\'source_int\', target=\'target_int\')[[\'source_int\', \'target_int\']].rename(columns={\'source_int\': \'source\', \'target_int\': \'target\'})\n    }\n\n    nodes = {\n        \'veins\': pd.DataFrame.from_dict(vein_pos).T.rename(columns={0: \'x\', 1: \'y\'}),\n        \'arteries\': pd.DataFrame.from_dict(artery_pos).T.rename(columns={0: \'x\', 1: \'y\'})\n    }\n\n    # Apply the hammer bundle method (assumed to be implemented elsewhere)\n    hb = {\n        \'veins\': hammer_bundle(nodes[\'veins\'], edges[\'veins\'], initial_bandwidth=.015, decay=0.8, tensi

In [ ]:
#this checks if there is a duplicate of "islet of Langerhans" as "endocrine pancreas"
# print(ftu_vessel[ftu_vessel["FTU"] == "endocrine pancreas"])

In [ ]:
%pip install "dask[complete]" scikit-image


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#plot_blood_vessels_with_bundling(G_fem_arteries, G_fem_veins, vein_pos_female, artery_pos_female, 'female')

In [ ]:
#plot_blood_vessels_with_bundling(G_male_arteries, G_male_veins, vein_pos_male, artery_pos_male, 'male')

# 4. Overlay the SVG figures using svgutils

In [ ]:
def flip_text_horizontally(input_file, output_file):
    # Parse the SVG file
    tree = ET.parse(input_file)
    root = tree.getroot()

    # Get the SVG namespace
    ns = {'svg': 'http://www.w3.org/2000/svg'}
    ET.register_namespace('', ns['svg'])  # Prevent unnecessary prefixes in output

    # Loop through all text elements in the SVG
    for text_element in root.findall('.//svg:text', ns):
        # Get the text element's position (x, y)
        x = float(text_element.attrib.get('x', '0'))
        y = float(text_element.attrib.get('y', '0'))

        # Apply a transformation to flip the text horizontally
        current_transform = text_element.attrib.get('transform', '')
        flip_transform = f"translate({2 * x}, 0) scale(-1, 1)"
        text_element.set('transform', f"{current_transform} {flip_transform}".strip())

    # Save the modified SVG
    tree.write(output_file)
    print(f"Text-flipped SVG saved to {output_file}")

In [ ]:
## Uncomment if flipping female wing.
# input_svg = f"{output_path}/vega_female_name_viz.svg"  # Path to your input SVG file
# output_svg = f"{output_path}/vega_female_name_viz_flipped.svg"
# flip_text_horizontally(input_svg, output_svg)

In [ ]:
## Use first line if flipping female wing.
# template = st.fromfile(f'{output_path}/vega_female_name_viz_flipped.svg')
template = st.fromfile(f'{output_path}/vega_female_name_viz.svg')
template.set_size(size=(str(canvas_size), str(canvas_size)))
#second_svg = st.fromfile(f'{output_path}/blood_viz_female_bundled.svg')
#second_svg.set_size(size=(str(canvas_size), str(canvas_size)))

In [ ]:
print("template exists:", "template" in globals())
print("second_svg exists:", "second_svg" in globals())

template exists: True
second_svg exists: False


In [ ]:
print(template.height, template.width) 
#not needed
#print(second_svg.height, second_svg.width) 

1720 1720


In [ ]:
#template.append(second_svg)
#template.save(f'{output_path}/female_butterfly_wing.svg')

In [ ]:
## Comment out if flipping female wing.
#input_svg = f"{output_path}/vega_male_name_viz.svg"  # Path to your input SVG file
#output_svg = f"{output_path}/vega_male_name_viz_flipped.svg"
#flip_text_horizontally(input_svg, output_svg)

In [ ]:
## Use first line if flipping female wing.
# template = st.fromfile(f'{output_path}/vega_male_name_viz.svg')
template = st.fromfile(f'{output_path}/vega_male_name_viz_flipped.svg')
template.set_size(size=(str(canvas_size), str(canvas_size)))
#second_svg = st.fromfile(f'{output_path}/blood_viz_male_bundled.svg')
#second_svg.set_size(size=(str(canvas_size), str(canvas_size)))

In [ ]:
print(template.height, template.width)

# Not needed for organ-specific visualizations
# print(second_svg.height, second_svg.width)

1720 1720


In [ ]:
#template.append(second_svg)
#template.save(f'{output_path}/male_butterfly_wing.svg')

# 5. In Adobe Illustrator
Place female_butterfly_wing.svg in AI:

X: 860

Y: 890

If female wing is to be flipped, Flip it horizontally,

Place male_butterfly_wing.svg:

X: 890

or X:830 if male wing is to be flipped.

Y: 890

If male wing is to be flipped, Flip it horizontally.
Add legend and Save file